# Import các thư viện cần thiết

In [ ]:
import pandas as pd
import numpy as np
import json
from datetime import datetime
from sklearn.preprocessing import StandardScaler, LabelEncoder #chuẩn hóa feature
from sklearn.metrics.pairwise import cosine_similarity #Cosin Similarity giữa 2 vector A và B có giá trị nằm trong khoảng [-1,1]. Càng gần 1 thì 2 user càng giống nhau
from sklearn.feature_extraction.text import TfidfVectorizer #vector hóa (thành vector số học có thể so sánh được)
from datetime import datetime
import warnings

In [67]:
warnings.filterwarnings('ignore')

# Đọc dữ liệu

In [68]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [69]:
real_profiles = pd.read_csv("/content/drive/MyDrive/Rcm-sys/Data/profile_rows.csv")

In [70]:
real_likes   = pd.read_csv("/content/drive/MyDrive/Rcm-sys/Data/likes_rows.csv")

In [71]:
real_matches   = pd.read_csv("/content/drive/MyDrive/Rcm-sys/Data/matches_rows.csv")

# Phân tích dữ liệu

In [72]:
real_profiles.head(5)

,id,username,gender,date_of_birth,bio,created_at,target_type,hobby,avatar_id,account_id,pref_min_age,pref_max_age,pref_gender,pref_location_city,height,zodiac_sign,job,education,location_name
0,1,Quang Anh dz,MALE,2003-12-01,HI iiiiiii tôi là quanh nè,2025-10-01 09:51:47+00,Một người bạn đời,"[""SINGING"",""READING"",""VIDEO_GAMES"",""FASHION"",""...",26.0,1,18,35,NaN,NaN,175.0,CAPRICORN,NaN,NaN,"Phường Khương Đình, Hà Nội"
1,2,Ngoc Mai,FEMALE,2005-12-06,hiiiiiiiiiiii,2025-10-02 15:28:37+00,Một người bạn đời,"[""YOGA"",""READING"",""PLAYING_GUITAR"",""RUNNING"",""...",20.0,2,18,35,NaN,NaN,176.0,ARIES,Tester,DH Bach Khoa HN,"Phường Khương Đình, Hà Nội"
2,3,Linh Mai,FEMALE,2004-06-15,Gaooo ồ,2025-10-07 14:07:34+00,Người yêu,"[""LISTENING_TO_MUSIC"",""YOGA"",""TRAVELING"",""VIDE...",37.0,3,18,35,NaN,NaN,170.0,CANCER,SPS,NaN,"Phường Khương Đình, Hà Nội"
3,4,Thanh Lam,FEMALE,2004-02-01,love u to the moon and back,2025-10-07 14:14:13+00,Người yêu,"[""READING"",""VIDEO_GAMES"",""TRAVELING"",""SINGING""...",25.0,4,18,35,NaN,NaN,NaN,NaN,NaN,NaN,"Phường Ô Chợ Dừa, Hà Nội"
4,5,Nam Tiến,MALE,2001-12-05,Im chicken guys,2025-10-11 02:02:14+00,Một người bạn đời,"[""LISTENING_TO_MUSIC"",""YOGA"",""TRAVELING"",""READ...",36.0,5,18,35,NaN,NaN,170.0,CAPRICORN,NaN,NaN,"Phường Khương Đình, Hà Nội"


# Xây dựng tập dữ liệu giả lập cho quá trình thử nghiệm

In [ ]:
def generate_fake_profiles(n_profiles=500, base_profiles_df=None): #Tạo dữ liệu fake profile
    """
    Tạo fake profiles dựa trên patterns từ real data

    Args:
        n_profiles: Số profiles cần tạo
        base_profiles_df: DataFrame profiles gốc để học pattern

    Returns:
        DataFrame với fake profiles
    """

    np.random.seed(42)

    # Vietnamese names
    male_first_names = [
      'Anh','Minh','Tuấn','Hùng','Đức','Nam','Khoa','Dũng','Phong','Long',
      'Quang','Tài','Hải','Việt','Thắng','Tiến','Hoàng','Bảo','Cường','Hưng',
      'Khánh','Phúc','Sơn','Lâm','Toàn','Vinh','Hiếu','Tín','Phát','Thịnh',
      'Quốc','Trường','Nhật','Bình','Tùng','An','Lộc','Đạt','Thành','Kiên',
      'Hào','Khôi','Nghĩa','Sỹ','Thiện','Trí','Huy','Chí','Công','Duy',
      'Luân','Phú','Sáng','Nhân','Tấn','Hậu','Đoàn','Vũ','Bách'
  ]

    last_names = [
      'Nguyễn','Trần','Lê','Phạm','Hoàng','Huỳnh','Phan','Vũ','Võ','Đặng',
      'Bùi','Đỗ','Hồ','Ngô','Dương','Lý','Đinh','Thái','Mai','Tạ',
      'Chung','Quách','Tôn','Lưu','Cao','Triệu','Hà','Châu','Tăng','La'
  ]

    female_first_names = [
      'Hoa','Mai','Lan','Hương','Linh','Thảo','Nga','Hà','Trang','Nhung',
      'Phương','Thu','Chi','Vy','My','Anh','Ngọc','Thanh','Kim','Châu',
      'Hạnh','Quỳnh','Yến','Trâm','Diễm','Tuyết','Nhi','Bích','Huyền','Ánh',
      'Tiên','Thúy','Dung','Hiền','Loan','Oanh','Lệ','Khanh','Tâm','An',
      'Giang','Liên','Thy','Thủy','Phúc','Uyên','Cúc','Mỹ','Na','Đào',
      'Lam','Sương','Vân','Xuân','Hồng','Nhàn','Tú','Ngân'
  ]

    hobbies = ['SINGING', 'READING', 'VIDEO_GAMES', 'FASHION', 'DOG_LOVER',
               'YOGA', 'PLAYING_GUITAR', 'RUNNING', 'LISTENING_TO_MUSIC',
               'TRAVELING', 'MEDITATION', 'COOKING', 'PHOTOGRAPHY', 'CYCLING',
               'PAINTING', 'WRITING', 'HIKING', 'SHOPPING', 'MOVIES', 'MUSIC']

    target_types = ['Người yêu', 'Một người bạn đời', 'Những người bạn mới', 'dating']

    zodiac_signs = ['ARIES', 'TAURUS', 'GEMINI', 'CANCER', 'LEO', 'VIRGO',
                   'LIBRA', 'SCORPIO', 'SAGITTARIUS', 'CAPRICORN',
                   'AQUARIUS', 'PISCES']

    cities = [
      'Phường Khương Đình, Hà Nội','Phường Ô Chợ Dừa, Hà Nội',
      'Quận Hai Bà Trưng, Hà Nội','Quận Đống Đa, Hà Nội','Quận Cầu Giấy, Hà Nội',
      'Quận Ba Đình, Hà Nội','Quận Hoàn Kiếm, Hà Nội','Quận Tây Hồ, Hà Nội',
      'Quận Long Biên, Hà Nội','Quận Hà Đông, Hà Nội',
      'Quận 1, TP.HCM','Quận 3, TP.HCM','Quận 4, TP.HCM','Quận 5, TP.HCM',
      'Quận Bình Thạnh, TP.HCM','Quận Phú Nhuận, TP.HCM',
      'Quận Tân Bình, TP.HCM','Quận Tân Phú, TP.HCM',
      'Quận Gò Vấp, TP.HCM','Quận Thủ Đức, TP.HCM',
      'Quận Hải Châu, Đà Nẵng','Quận Thanh Khê, Đà Nẵng',
      'Quận Sơn Trà, Đà Nẵng','Quận Liên Chiểu, Đà Nẵng',
      'TP Thủ Dầu Một, Bình Dương','TP Biên Hòa, Đồng Nai',
      'TP Nha Trang, Khánh Hòa','TP Vũng Tàu, Bà Rịa - Vũng Tàu',
      'TP Huế, Thừa Thiên Huế','TP Đà Lạt, Lâm Đồng',
      'TP Cần Thơ','TP Rạch Giá, Kiên Giang'
  ]

    jobs = [
      'Developer','Software Engineer','Frontend Developer','Backend Developer',
      'Designer','UI/UX Designer','Graphic Designer',
      'Marketing','Digital Marketing','Content Creator','SEO Specialist',
      'Teacher','Lecturer','Tutor',
      'Engineer','Civil Engineer','Mechanical Engineer','Electrical Engineer',
      'Doctor','Dentist','Pharmacist','Nurse',
      'Accountant','Auditor','Financial Analyst',
      'Manager','Project Manager','Product Manager',
      'HR','Recruiter',
      'Sales','Business Development',
      'Student','Graduate Student',
      'Freelancer','Photographer','Videographer',
      'Data Analyst','Data Scientist','AI Engineer',
      'QA Engineer','Tester','DevOps Engineer',
      'Consultant','Researcher'
  ]

    educations = [
      'DH Bach Khoa HN','DH Quoc Gia HN','DH Kinh Te Quoc Dan',
      'DH Thuong Mai','DH Ngoai Thuong','DH FPT','RMIT Vietnam',
      'DH Y Ha Noi','DH Duoc Ha Noi',
      'DH KHTN','DH KHXH&NV','DH Su Pham',
      'DH Giao Thong Van Tai','DH Xay Dung',
      'DH Cong Nghiep','DH Cong Nghe',
      'DH Ton Duc Thang','DH Bach Khoa TP.HCM',
      'DH Khoa Hoc Tu Nhien TP.HCM',
      'DH Quoc Gia TP.HCM',
      'Cao Dang FPT','Cao Dang Du Lich',
      'Trung Cap Nghe',
      'Dang hoc Dai hoc','Tot nghiep THPT',
      'Du hoc sinh','Thac si','Tien si'
  ]


    bio_templates = [
      "Hi! Tôi là {name}",
      "Xin chào, rất vui được làm quen",
      "Một người bình thường với cuộc sống thú vị",
      "Tìm người tâm đầu ý hợp",
      "Muốn tìm một người có thể cùng chia sẻ mọi thứ",
      "Yêu du lịch và khám phá",
      "Thích những chuyến đi ngẫu hứng",
      "Sống tích cực, yêu cuộc sống",
      "Luôn tin vào những điều tốt đẹp",
      "Coffee lover ☕",
      "Thích cà phê và những buổi chiều yên tĩnh",
      "Thích đọc sách và nghe nhạc",
      "Âm nhạc là một phần không thể thiếu",
      "Yêu thiên nhiên và những điều giản dị",
      "Cuộc sống là những trải nghiệm",
      "Just be yourself",
      "Enjoy life to the fullest",
      "Looking for someone special",
      "Love traveling and adventures",
      "Simple life, happy life",
      "Chậm lại một chút để cảm nhận cuộc sống",
      "Thích nói chuyện sâu và meaningful",
      "Hướng nội nhưng không ngại trò chuyện",
      "Hướng ngoại, vui vẻ và hòa đồng",
      "Một người yêu gia đình",
      "Yêu chó và động vật 🐶",
      "Đam mê nấu ăn và thử món mới",
      "Thích thể thao và vận động",
      "Chạy bộ giúp mình cân bằng cuộc sống",
      "Yoga mỗi sáng, cà phê mỗi chiều",
      "Thích xem phim cuối tuần",
      "Một người thích sự đơn giản",
      "Không hoàn hảo nhưng chân thành",
      "Tìm một người để cùng già đi",
      "Muốn tìm bạn mới, không áp lực",
      "Nghiêm túc trong mối quan hệ",
      "Không vội, mọi thứ để tự nhiên",
      "Let’s see where this goes",
      "Here to make new connections",
      "Chỉ là chính mình thôi",
      "Mỗi ngày là một hành trình mới",
      "Yêu những điều nhỏ bé",
      "Cười nhiều hơn, lo ít hơn",
      "Tin vào duyên phận",
      "Một người sống tình cảm",
      "Thích những buổi trò chuyện khuya",
      "Thích biển hơn núi",
      "Yêu hoàng hôn và cà phê sữa",
      "Cần một người đủ kiên nhẫn",
      "Một người đang học cách yêu bản thân",
      "Thích sự chân thành hơn mọi thứ",
      "Cuộc sống không chỉ là công việc",
      "Thích đi bộ và suy nghĩ",
      "Một người đang tìm sự đồng cảm",
      "Chào bạn 👋",
      "Một người bình thường",
      "Một người đơn giản",
      "New here",
      "Lần đầu dùng app",
      "Muốn tìm một người để trò chuyện",
      "Tìm một mối quan hệ nghiêm túc",
      "Muốn tìm người phù hợp",
      "Tin rằng duyên sẽ tới",
      "Hy vọng gặp được người phù hợp",
      "Mở lòng với những mối quan hệ mới",
      "Yêu sự chân thành",
      "Thích sự đơn giản",
      "Chậm lại để cảm nhận cuộc sống",
      "Một người hướng nội",
      "Hướng ngoại và hòa đồng",
      "Thích nói chuyện sâu",
      "Âm nhạc là một phần cuộc sống",
      "Yêu thiên nhiên",
      "Thích nấu ăn và thử món mới",
      "Yêu hoàng hôn và biển",
      "Tìm một người có thể chia sẻ mọi thứ",
      "Muốn tìm người để cùng trưởng thành",
      "Một người nghiêm túc trong tình cảm",
      "Muốn tìm người đồng cảm",
      "Tin vào sự thấu hiểu",
      "Open to new experiences",
      "Living one day at a time",
      "Trying to enjoy the little things in life",
      "Looking for meaningful conversations",
      "Life is better when shared",
      "Not perfect, but always genuine",
      "A calm mind and a warm heart",
      "Here for real connections, not games",
      "Believe in timing and honesty",
      "Sống chậm lại và enjoy cuộc sống",
      "Một người simple nhưng sincere",
      "Yêu sự bình yên và good vibes",
      "Just chill and be happy",
      "Cuộc sống là balance",
      "Hãy để chúng ta cùng nhau viết câu chuyện",
      "Nếu bạn muốn trò chuyện, cứ nhắn nhé",
      "Biết đâu chúng ta hợp nhau",
      "Cứ nói chuyện rồi xem sao",
      "Mọi thứ bắt đầu từ một cuộc trò chuyện"
  ]

    fake_data = []
    start_id = 1000 if base_profiles_df is None else base_profiles_df['id'].max() + 1

    for i in range(n_profiles):
        profile_id = start_id + i
        gender = np.random.choice(['MALE', 'FEMALE'], p=[0.55, 0.45]) #55% nam, 45% nữ

        if gender == 'MALE':
            first_name = np.random.choice(male_first_names)
            last_name = np.random.choice(last_names)
            username = f"{first_name} {last_name}"
        else:
            first_name = np.random.choice(female_first_names)
            last_name = np.random.choice(last_names)
            username = f"{first_name} {last_name}"

        # Age distribution: mostly 20-30
        age = int(np.random.normal(24, 3)) #Tập trung quanh 24 tuổi
        age = np.clip(age, 18, 40)
        birth_year = 2026 - age
        birth_month = np.random.randint(1, 13)
        birth_day = np.random.randint(1, 29)
        date_of_birth = f"{birth_year}-{birth_month:02d}-{birth_day:02d}"

        # Bio
        bio_template = np.random.choice(bio_templates)
        bio = bio_template.format(name=first_name) if '{name}' in bio_template else bio_template

        # Created date (last 100 days)
        days_ago = np.random.randint(1, 100)
        created_at = pd.Timestamp.now(tz='UTC') - pd.Timedelta(days=days_ago)
        created_at_str = created_at.strftime('%Y-%m-%d %H:%M:%S+00')

        # Target type
        target_type = np.random.choice(target_types, p=[0.3, 0.3, 0.2, 0.2])

        # Hobbies (2-5 hobbies từ list 20 hobbies) 
        n_hobbies = np.random.randint(2, 6)
        user_hobbies = np.random.choice(hobbies, n_hobbies, replace=False)
        hobby_str = str(list(user_hobbies)).replace("'", '"')

        # Avatar ID
        avatar_id = int(np.random.randint(1, 100))

        # Account ID
        account_id = profile_id

        # Preferences (80% set preferences)
        if np.random.random() < 0.8:
            pref_min_age = int(max(18, age - 5))
            pref_max_age = int(min(40, age + 7))
            pref_gender = 'FEMALE' if gender == 'MALE' else 'MALE'
        else:
            pref_min_age = np.nan
            pref_max_age = np.nan
            pref_gender = np.nan

        pref_location_city = np.nan

        # Height (80% fill)
        if np.random.random() < 0.8:
            if gender == 'MALE':
                height = int(np.random.normal(172, 6))
            else:
                height = int(np.random.normal(160, 5))
            height = round(height, 1)
        else:
            height = np.nan

        # Zodiac (70% fill)
        zodiac_sign = np.random.choice(zodiac_signs) if np.random.random() < 0.7 else np.nan

        # Job (60% fill)
        job = np.random.choice(jobs) if np.random.random() < 0.6 else np.nan

        # Education (50% fill)
        education = np.random.choice(educations) if np.random.random() < 0.5 else np.nan

        # Location
        location_name = np.random.choice(cities)

        fake_data.append({
            'id': profile_id,
            'username': username,
            'gender': gender,
            'date_of_birth': date_of_birth,
            'bio': bio,
            'created_at': created_at_str,
            'target_type': target_type,
            'hobby': hobby_str,
            'avatar_id': avatar_id,
            'account_id': account_id,
            'pref_min_age': pref_min_age,
            'pref_max_age': pref_max_age,
            'pref_gender': pref_gender,
            'pref_location_city': pref_location_city,
            'height': height,
            'zodiac_sign': zodiac_sign,
            'job': job,
            'education': education,
            'location_name': location_name
        })

    fake_df = pd.DataFrame(fake_data)

    print(f"✅ Đã tạo {len(fake_df)} fake profiles")
    print(f"   - Nam: {(fake_df['gender']=='MALE').sum()} ({(fake_df['gender']=='MALE').mean()*100:.1f}%)")
    print(f"   - Nữ: {(fake_df['gender']=='FEMALE').sum()} ({(fake_df['gender']=='FEMALE').mean()*100:.1f}%)")
    print(f"   - Tuổi trung bình: {(2026 - pd.to_datetime(fake_df['date_of_birth']).dt.year).mean():.1f}")

    return fake_df


In [74]:
real_profiles.columns

Index(['id', 'username', 'gender', 'date_of_birth', 'bio', 'created_at',
       'target_type', 'hobby', 'avatar_id', 'account_id', 'pref_min_age',
       'pref_max_age', 'pref_gender', 'pref_location_city', 'height',
       'zodiac_sign', 'job', 'education', 'location_name'],
      dtype='object')

In [ ]:
def _parse_hobby_list(hobby_str): #chuẩn háo dữ liệu thô
    # hobby_str kiểu '["A","B"]'
    try:
        return set(ast.literal_eval(hobby_str))
    except Exception:
        return set()

def _softmax(x, temp=1.0):
    x = np.array(x, dtype=float)
    x = x / max(temp, 1e-6)
    x = x - np.max(x)  # ổn định số
    e = np.exp(x)
    s = e.sum()
    return e / s if s > 0 else np.ones_like(e) / len(e)

In [ ]:
def generate_fake_likes(profiles_df, n_likes=5000): #lặp 5000 lần 
    """
    Tạo fake likes dựa trên logic realistic

    Args:
        profiles_df: DataFrame profiles
        n_likes: Số likes cần tạo

    Returns:
        DataFrame với fake likes
    """
    print(f"\n{'='*70}")
    print(f"TẠO {n_likes} FAKE LIKES")
    print(f"{'='*70}\n")

    np.random.seed(42)

    profile_ids = profiles_df['id'].tolist()
    fake_likes = []
    like_id = 1

    # Tạo likes với logic realistic Chỉ like người phù hợp preference (giới tính, tuổi)
    for _ in range(n_likes):
        # Random liker
        liker_id = np.random.choice(profile_ids)
        liker_row = profiles_df[profiles_df['id'] == liker_id].iloc[0]

        # Filter candidates theo preferences
        candidates = profiles_df.copy()

        # Filter by gender preference 
        if pd.notna(liker_row['pref_gender']):
            candidates = candidates[candidates['gender'] == liker_row['pref_gender']]

        # Filter by age preference
        if pd.notna(liker_row['pref_min_age']):
            candidates['age'] = 2026 - pd.to_datetime(candidates['date_of_birth']).dt.year
            candidates = candidates[candidates['age'] >= liker_row['pref_min_age']]

        if pd.notna(liker_row['pref_max_age']):
            if 'age' not in candidates.columns:
                candidates['age'] = 2026 - pd.to_datetime(candidates['date_of_birth']).dt.year
            candidates = candidates[candidates['age'] <= liker_row['pref_max_age']]

        # Remove self
        candidates = candidates[candidates['id'] != liker_id]

        if len(candidates) == 0:
            continue

        # Pick random liked
        liked_id = np.random.choice(candidates['id'].tolist())

        # Check duplicate
        if any(l['liker_id'] == liker_id and l['liked_id'] == liked_id for l in fake_likes):
            continue

        # Random timestamp (last 90 days)
        days_ago = np.random.randint(0, 90)
        hours_ago = np.random.randint(0, 24)
        minutes_ago = np.random.randint(0, 60)

        timestamp = pd.Timestamp.now(tz='UTC') - pd.Timedelta(
            days=days_ago, hours=hours_ago, minutes=minutes_ago
        )
        timestamp_str = timestamp.strftime('%Y-%m-%d %H:%M:%S')

        fake_likes.append({
            'id': like_id,
            'liker_id': liker_id,
            'liked_id': liked_id,
            'timestamp': timestamp_str
        })
        like_id += 1

    fake_likes_df = pd.DataFrame(fake_likes)

    print(f"✅ Đã tạo {len(fake_likes_df)} fake likes")
    print(f"   - Unique likers: {fake_likes_df['liker_id'].nunique()}")
    print(f"   - Unique liked: {fake_likes_df['liked_id'].nunique()}")
    print(f"   - Avg likes per user: {len(fake_likes_df) / fake_likes_df['liker_id'].nunique():.1f}")

    return fake_likes_df


In [ ]:
def generate_fake_matches(likes_df, match_rate=0.15): #15% mutual likes thành match
    """
    Tạo fake matches từ likes (mutual likes)

    Args:
        likes_df: DataFrame likes
        match_rate: Tỷ lệ likes thành match

    Returns:
        DataFrame với fake matches
    """
    print(f"\n{'='*70}")
    print(f"TẠO FAKE MATCHES (Match rate: {match_rate*100:.0f}%)")
    print(f"{'='*70}\n")

    np.random.seed(42)

    fake_matches = []
    match_id = 1

    # Tạo dict để lookup nhanh
    likes_dict = {}
    for _, row in likes_df.iterrows():
        liker = row['liker_id']
        liked = row['liked_id']
        if liker not in likes_dict:
            likes_dict[liker] = set()
        likes_dict[liker].add(liked)

    # Tìm mutual likes
    processed_pairs = set()

    for _, row in likes_df.iterrows():
        user1 = row['liker_id']
        user2 = row['liked_id']

        # Check if already processed
        pair = tuple(sorted([user1, user2]))
        if pair in processed_pairs:
            continue

        # Check if mutual like
        if user2 in likes_dict.get(user1, set()) and user1 in likes_dict.get(user2, set()):
            # Random match với probability
            if np.random.random() < match_rate:
                # Match timestamp = sau timestamp của like thứ 2
                like1_time = likes_df[
                    (likes_df['liker_id'] == user1) & (likes_df['liked_id'] == user2)
                ]['timestamp'].iloc[0]
                like2_time = likes_df[
                    (likes_df['liker_id'] == user2) & (likes_df['liked_id'] == user1)
                ]['timestamp'].iloc[0]

                match_time = max(pd.to_datetime(like1_time), pd.to_datetime(like2_time))
                match_time += pd.Timedelta(seconds=np.random.randint(1, 300))  # 1-5 minutes later

                fake_matches.append({
                    'id': match_id,
                    'user1_id': user1,
                    'user2_id': user2,
                    'matched_at': match_time.strftime('%Y-%m-%d %H:%M:%S')
                })
                match_id += 1
                processed_pairs.add(pair)

    fake_matches_df = pd.DataFrame(fake_matches)

    print(f"✅ Đã tạo {len(fake_matches_df)} fake matches")
    if len(fake_matches_df) > 0:
        print(f"   - Unique users in matches: {pd.concat([fake_matches_df['user1_id'], fake_matches_df['user2_id']]).nunique()}")
        print(f"   - Actual match rate: {len(fake_matches_df) / len(likes_df) * 100:.2f}%")

    return fake_matches_df


In [78]:
def combine_real_and_fake_data(real_df, fake_df):
    """
    Kết hợp real data và fake data

    Args:
        real_df: DataFrame real data
        fake_df: DataFrame fake data

    Returns:
        DataFrame combined
    """
    print(f"\n{'='*70}")
    print("KẾT HỢP REAL & FAKE DATA")
    print(f"{'='*70}\n")

    combined = pd.concat([real_df, fake_df], ignore_index=True)

    print(f"✅ Kết hợp thành công!")
    print(f"   - Real data: {len(real_df)} rows")
    print(f"   - Fake data: {len(fake_df)} rows")
    print(f"   - Total: {len(combined)} rows")

    return combined

# Hàm tăng cường dữ liệu

In [79]:
def calculate_age(profiles_df):
    """
    Tính tuổi từ date_of_birth

    Args:
        profiles_df: DataFrame chứa cột 'date_of_birth'

    Returns:
        DataFrame với cột 'age' mới
    """
    print("🔄 Đang tính tuổi từ date_of_birth...")
    df = profiles_df.copy()
    df['date_of_birth'] = pd.to_datetime(df['date_of_birth'])
    current_year = datetime.now().year
    df['age'] = current_year - df['date_of_birth'].dt.year
    print(f"✅ Đã tính tuổi cho {len(df)} profiles")
    print(f"   - Tuổi trung bình: {df['age'].mean():.1f}")
    print(f"   - Tuổi min-max: {df['age'].min()}-{df['age'].max()}")
    return df

In [80]:
def parse_hobby_features(profiles_df):
    """
    Parse hobby từ string JSON thành list và tính các features liên quan

    Args:
        profiles_df: DataFrame chứa cột 'hobby'

    Returns:
        DataFrame với 'hobby_list' và 'hobby_count'
    """
    print("\n🔄 Đang parse hobby features...")
    df = profiles_df.copy()

    def parse_hobby(hobby_str):
        try:
            if pd.isna(hobby_str):
                return []
            hobbies = hobby_str.strip('[]"').replace('"', '').split(',')
            return [h.strip() for h in hobbies if h.strip()]
        except:
            return []

    df['hobby_list'] = df['hobby'].apply(parse_hobby)
    df['hobby_count'] = df['hobby_list'].apply(len)

    print(f"✅ Đã parse hobby cho {len(df)} profiles")
    print(f"   - Số hobby trung bình: {df['hobby_count'].mean():.1f}")
    print(f"   - Max hobbies: {df['hobby_count'].max()}")
    return df

In [81]:
def create_bio_features(profiles_df):
    """
    Tạo các features từ bio

    Args:
        profiles_df: DataFrame chứa cột 'bio'

    Returns:
        DataFrame với 'bio_length' và 'has_bio'
    """
    print("\n🔄 Đang tạo bio features...")
    df = profiles_df.copy()
    df['bio_length'] = df['bio'].fillna('').str.len()
    df['has_bio'] = (df['bio_length'] > 0).astype(int)

    print(f"✅ Đã tạo bio features")
    print(f"   - % có bio: {df['has_bio'].mean()*100:.1f}%")
    print(f"   - Độ dài bio trung bình: {df['bio_length'].mean():.0f} ký tự")
    return df

In [82]:
def calculate_account_age(profiles_df):
    """
    Tính số ngày từ khi tạo tài khoản

    Args:
        profiles_df: DataFrame chứa cột 'created_at'

    Returns:
        DataFrame với 'account_age_days'
    """
    print("\n🔄 Đang tính tuổi tài khoản...")
    df = profiles_df.copy()

    # Parse với format='mixed' để handle nhiều formats khác nhau
    df['created_at'] = pd.to_datetime(df['created_at'], format='mixed', utc=True)

    # Convert current time to UTC để tính toán đúng
    now_utc = pd.Timestamp.now(tz='UTC')
    df['account_age_days'] = (now_utc - df['created_at']).dt.days

    print(f"✅ Đã tính tuổi tài khoản")
    print(f"   - Tuổi tài khoản trung bình: {df['account_age_days'].mean():.0f} ngày")
    print(f"   - Tuổi tài khoản min-max: {df['account_age_days'].min()}-{df['account_age_days'].max()} ngày")
    return df

In [83]:
def calculate_profile_completeness(profiles_df):
    """
    Tính điểm hoàn thiện profile (0-1)

    Args:
        profiles_df: DataFrame với các cột profile

    Returns:
        DataFrame với 'profile_completeness'
    """
    print("\n🔄 Đang tính profile completeness...")
    df = profiles_df.copy()
    completeness_cols = ['height', 'zodiac_sign', 'job', 'education', 'bio']
    df['profile_completeness'] = df[completeness_cols].notna().sum(axis=1) / len(completeness_cols)

    print(f"✅ Đã tính profile completeness")
    print(f"   - Completeness trung bình: {df['profile_completeness'].mean()*100:.1f}%")
    return df

In [84]:
def calculate_engagement_metrics(profiles_df, likes_df, matches_df):
    """
    Tính các engagement metrics từ likes và matches

    Args:
        profiles_df: DataFrame profiles
        likes_df: DataFrame likes
        matches_df: DataFrame matches

    Returns:
        DataFrame với engagement metrics
    """
    print("\n🔄 Đang tính engagement metrics...")
    df = profiles_df.copy()

    # Likes given và received
    likes_given = likes_df.groupby('liker_id').size().rename('likes_given_count')
    likes_received = likes_df.groupby('liked_id').size().rename('likes_received_count')

    # Matches count
    matches_count = pd.concat([
        matches_df.groupby('user1_id').size(),
        matches_df.groupby('user2_id').size()
    ]).groupby(level=0).sum().rename('matches_count')

    # Merge vào profiles
    df = df.merge(likes_given, left_on='id', right_index=True, how='left')
    df = df.merge(likes_received, left_on='id', right_index=True, how='left')
    df = df.merge(matches_count, left_on='id', right_index=True, how='left')

    # Fill NA với 0
    df[['likes_given_count', 'likes_received_count', 'matches_count']] = \
        df[['likes_given_count', 'likes_received_count', 'matches_count']].fillna(0)

    # Match rate
    df['match_rate'] = np.where(
        df['likes_given_count'] > 0,
        df['matches_count'] / df['likes_given_count'],
        0
    )

    print(f"✅ Đã tính engagement metrics")
    print(f"   - Avg likes given: {df['likes_given_count'].mean():.1f}")
    print(f"   - Avg likes received: {df['likes_received_count'].mean():.1f}")
    print(f"   - Avg matches: {df['matches_count'].mean():.1f}")
    print(f"   - Avg match rate: {df['match_rate'].mean()*100:.1f}%")
    return df

In [85]:
def calculate_engagement_metrics(profiles_df, likes_df, matches_df):
    """
    Tính các engagement metrics từ likes và matches

    Args:
        profiles_df: DataFrame profiles
        likes_df: DataFrame likes
        matches_df: DataFrame matches

    Returns:
        DataFrame với engagement metrics
    """
    print("\n🔄 Đang tính engagement metrics...")
    df = profiles_df.copy()

    # Likes given và received
    likes_given = likes_df.groupby('liker_id').size().rename('likes_given_count')
    likes_received = likes_df.groupby('liked_id').size().rename('likes_received_count')

    # Matches count
    matches_count = pd.concat([
        matches_df.groupby('user1_id').size(),
        matches_df.groupby('user2_id').size()
    ]).groupby(level=0).sum().rename('matches_count')

    # Merge vào profiles
    df = df.merge(likes_given, left_on='id', right_index=True, how='left')
    df = df.merge(likes_received, left_on='id', right_index=True, how='left')
    df = df.merge(matches_count, left_on='id', right_index=True, how='left')

    # Fill NA với 0
    df[['likes_given_count', 'likes_received_count', 'matches_count']] = \
        df[['likes_given_count', 'likes_received_count', 'matches_count']].fillna(0)

    # Match rate
    df['match_rate'] = np.where(
        df['likes_given_count'] > 0,
        df['matches_count'] / df['likes_given_count'],
        0
    )

    print(f"✅ Đã tính engagement metrics")
    print(f"   - Avg likes given: {df['likes_given_count'].mean():.1f}")
    print(f"   - Avg likes received: {df['likes_received_count'].mean():.1f}")
    print(f"   - Avg matches: {df['matches_count'].mean():.1f}")
    print(f"   - Avg match rate: {df['match_rate'].mean()*100:.1f}%")
    return df

In [86]:
def normalize_location(profiles_df):
    """
    Chuẩn hóa location

    Args:
        profiles_df: DataFrame với cột 'location_name'

    Returns:
        DataFrame với 'city'
    """
    print("\n🔄 Đang chuẩn hóa location...")
    df = profiles_df.copy()
    df['city'] = df['location_name'].fillna('Unknown')

    print(f"✅ Đã chuẩn hóa location")
    print(f"   - Số thành phố unique: {df['city'].nunique()}")
    return df

In [ ]:
def enrich_all_features(profiles_df, likes_df, matches_df): #Biến raw data thành features có thể dùng cho Model
    """
    Chạy tất cả các bước feature engineering

    Args:
        profiles_df: DataFrame profiles gốc
        likes_df: DataFrame likes
        matches_df: DataFrame matches

    Returns:
        DataFrame profiles đã được làm giàu
    """
    print("="*70)
    print("BẮT ĐẦU FEATURE ENGINEERING")
    print("="*70)

    # Apply tất cả transformations
    df = profiles_df.copy()
    df = calculate_age(df) #Tuổi từ date_of_birth
    df = parse_hobby_features(df) #Parse hobbies từ string → list
    df = create_bio_features(df) 
    df = calculate_account_age(df) # Số ngày từ khi tạo tài khoản
    df = calculate_profile_completeness(df) # điểm hoàn thiện profile
    df = calculate_engagement_metrics(df, likes_df, matches_df)
    df = normalize_location(df)

    print("\n" + "="*70)
    print("HOÀN THÀNH FEATURE ENGINEERING")
    print("="*70)
    print(f"\n📊 Thống kê:")
    print(f"   - Tổng số profiles: {len(df)}")
    print(f"   - Tổng số features: {len(df.columns)}")
    print(f"\n✨ Các features mới đã tạo:")
    new_features = ['age', 'hobby_list', 'hobby_count', 'bio_length', 'has_bio',
                   'account_age_days', 'profile_completeness', 'likes_given_count',
                   'likes_received_count', 'matches_count', 'match_rate', 'city']
    for i, feat in enumerate(new_features, 1):
        print(f"   {i:2d}. {feat}")

    return df


In [88]:
def save_enriched_data(enriched_df, output_path="enriched_profiles.csv"):
    """
    Lưu dữ liệu đã làm giàu ra file CSV

    Args:
        enriched_df: DataFrame đã được làm giàu
        output_path: Đường dẫn file output
    """
    print(f"\n💾 Đang lưu dữ liệu vào {output_path}...")
    enriched_df.to_csv(output_path, index=False)
    print(f"✅ Đã lưu thành công!")
    return output_path

In [ ]:
class DatingRecommendationSystem:
    """
    Hệ thống khuyến nghị cho dating app
    Kết hợp Content-based và Collaborative Filtering
    """

    def __init__(self, enriched_profiles_df, likes_df, matches_df):
        """
        Khởi tạo hệ thống recommendation

        Args:
            enriched_profiles_df: DataFrame profiles đã được làm giàu
            likes_df: DataFrame likes
            matches_df: DataFrame matches
        """
        print("\n" + "="*70)
        print("KHỞI TẠO RECOMMENDATION SYSTEM")
        print("="*70)

        self.profiles = enriched_profiles_df.copy()
        self.likes = likes_df.copy()
        self.matches = matches_df.copy()

        # Prepare features
        self._prepare_content_features()
        self._prepare_collaborative_features()

        print("\n✅ Khởi tạo thành công!")

    def _filter_candidates_by_prefs(self, user_row, candidates_df):
        df = candidates_df.copy()

        # Gender preference
        pref_gender = user_row.get('pref_gender', np.nan)
        if pd.notna(pref_gender) and pref_gender != 'BOTH':
            df = df[df['gender'] == pref_gender]

        # Age preference (ưu tiên age_raw)
        age_col = 'age_raw' if 'age_raw' in df.columns else 'age'
        if pd.notna(user_row.get('pref_min_age', np.nan)):
            df = df[df[age_col] >= user_row['pref_min_age']]
        if pd.notna(user_row.get('pref_max_age', np.nan)):
            df = df[df[age_col] <= user_row['pref_max_age']]

        return df


    def _prepare_content_features(self): #Chuẩn bị features cho Content-based Filtering
      """Chuẩn bị features cho Content-based Filtering"""
      print("\n🔄 Đang chuẩn bị Content-based features...")

      colmap = self._get_profile_cols()
      username_col = colmap['username']
      city_col = colmap['city']

      # Nếu city không có -> tạo city = Unknown
      if city_col is None:
          self.profiles['city'] = 'Unknown'
      else:
          # chuẩn hoá về cột city để dùng thống nhất
          self.profiles['city'] = self.profiles[city_col].fillna('Unknown')

      # BẮT BUỘC: phải có age raw để filter
      if 'age' not in self.profiles.columns:
          # nếu chưa có age mà có date_of_birth
          if 'date_of_birth' in self.profiles.columns:
              dob = pd.to_datetime(self.profiles['date_of_birth'], errors='coerce')
              self.profiles['age'] = 2026 - dob.dt.year
          else:
              raise KeyError("Không có 'age' hoặc 'date_of_birth' để tính age")

      # ✅ GIỮ age thật để filter/hiển thị
      self.profiles['age_raw'] = self.profiles['age']

      # Encode categorical features
      self.le_gender = LabelEncoder()
      self.le_target = LabelEncoder()
      self.le_zodiac = LabelEncoder()
      self.le_city = LabelEncoder()

      self.profiles['gender_encoded'] = self.le_gender.fit_transform( #Chuyển "MALE"/"FEMALE" thành 0/1
          self.profiles['gender'].fillna('UNKNOWN')
      )
      self.profiles['target_type_encoded'] = self.le_target.fit_transform(
          self.profiles['target_type'].fillna('UNKNOWN')
      )
      self.profiles['zodiac_encoded'] = self.le_zodiac.fit_transform(
          self.profiles['zodiac_sign'].fillna('UNKNOWN')
      )
      self.profiles['city_encoded'] = self.le_city.fit_transform(
          self.profiles['city'].fillna('Unknown')
      )

      # Numerical columns (giữ raw + tạo scaled)
      numerical_cols = ['age_raw', 'height', 'hobby_count', 'profile_completeness', 'account_age_days']
      for col in numerical_cols:
          if col not in self.profiles.columns:
              self.profiles[col] = np.nan
          self.profiles[col] = self.profiles[col].fillna(self.profiles[col].median())

      self.scaler = StandardScaler() #Chuẩn hóa age, height, hobby_count về mean=0, std=1
      scaled = self.scaler.fit_transform(self.profiles[numerical_cols])

      scaled_cols = [c.replace('_raw', '') + '_scaled' for c in numerical_cols]
      for i, c in enumerate(scaled_cols):
          self.profiles[c] = scaled[:, i]

      # Hobby TF-IDF
      if 'hobby_list' not in self.profiles.columns:
          # fallback nếu bạn lưu hobby dạng string list
          if 'hobby' in self.profiles.columns:
              # cố parse đơn giản
              def _parse_h(x):
                  try:
                      if isinstance(x, str):
                          x2 = x.replace("'", '"')
                          return list(pd.read_json(pd.Series([x2]))[0])
                  except:
                      return []
                  return []
              self.profiles['hobby_list'] = self.profiles['hobby'].apply(_parse_h)
          else:
              self.profiles['hobby_list'] = [[] for _ in range(len(self.profiles))]

      self.profiles['hobby_text'] = self.profiles['hobby_list'].apply(
          lambda x: ' '.join(x) if isinstance(x, list) and len(x) > 0 else ''
      )

      self.tfidf_hobby = TfidfVectorizer(max_features=50) #Biến hobbies thành vectors số học
      hobby_matrix = self.tfidf_hobby.fit_transform(self.profiles['hobby_text'])

      # Combine all features (dùng scaled columns)
      content_features = [
          'gender_encoded', 'target_type_encoded', 'zodiac_encoded', 'city_encoded'
      ] + scaled_cols

      self.content_feature_matrix = np.hstack([ #Combine
          self.profiles[content_features].values,
          hobby_matrix.toarray()
      ])

      print(f"✅ Content feature matrix: {self.content_feature_matrix.shape}")


    def _prepare_collaborative_features(self):
        """Chuẩn bị features cho Collaborative Filtering"""
        print("\n🔄 Đang chuẩn bị Collaborative features...")

        # Create user mapping
        user_ids = self.profiles['id'].unique()
        self.user_id_to_idx = {uid: idx for idx, uid in enumerate(user_ids)}
        self.idx_to_user_id = {idx: uid for uid, idx in self.user_id_to_idx.items()}

        # Create interaction matrix
        n_users = len(user_ids)
        self.interaction_matrix = np.zeros((n_users, n_users)) #Tạo interaction matrix

        for _, row in self.likes.iterrows():
            liker_id = row['liker_id']
            liked_id = row['liked_id']

            if liker_id in self.user_id_to_idx and liked_id in self.user_id_to_idx:
                liker_idx = self.user_id_to_idx[liker_id]
                liked_idx = self.user_id_to_idx[liked_id]
                self.interaction_matrix[liker_idx, liked_idx] = 1

        # Calculate user similarity: So sánh interaction patterns =>> Tính user similarity
        self.user_similarity_matrix = cosine_similarity(self.interaction_matrix)

        print(f"✅ Interaction matrix: {self.interaction_matrix.shape}")
        print(f"✅ Tổng interactions: {int(self.interaction_matrix.sum())}")

    def get_content_recommendations(self, user_id, top_n=10):
      """
      Gợi ý dựa trên Content-based Filtering

      Args:
          user_id: ID của user
          top_n: Số lượng recommendations

      Returns:
          DataFrame recommendations
      """
      if user_id not in self.profiles['id'].values:
          print(f"❌ User {user_id} không tồn tại!")
          return pd.DataFrame()

      # ✅ đảm bảo có age_raw (tuổi thật) để filter/hiển thị
      if 'age_raw' not in self.profiles.columns:
          if 'age' in self.profiles.columns:
              self.profiles['age_raw'] = self.profiles['age']
          else:
              raise KeyError("Không có cột 'age_raw' hoặc 'age' trong profiles")

      # Get user info
      user_idx = self.profiles[self.profiles['id'] == user_id].index[0]
      user_row = self.profiles.loc[user_idx]

      # Calculate similarity (Cosine similarity giữa user và tất cả users khác)
      user_features = self.content_feature_matrix[user_idx].reshape(1, -1)
      similarities = cosine_similarity(user_features, self.content_feature_matrix)[0]

      # Apply filters (Chỉ recommend người phù hợp preferences)
      valid_indices = np.arange(len(self.profiles))

      # Gender preference
      if pd.notna(user_row.get('pref_gender', np.nan)) and user_row['pref_gender'] != 'BOTH':
          valid_indices = valid_indices[
              self.profiles.iloc[valid_indices]['gender'] == user_row['pref_gender']
          ]

      # ✅ Age preference (dùng age_raw thay vì age đã bị scale)
      if pd.notna(user_row.get('pref_min_age', np.nan)):
          valid_indices = valid_indices[
              self.profiles.iloc[valid_indices]['age_raw'] >= user_row['pref_min_age']
          ]
      if pd.notna(user_row.get('pref_max_age', np.nan)):
          valid_indices = valid_indices[
              self.profiles.iloc[valid_indices]['age_raw'] <= user_row['pref_max_age']
          ]

      # Exclude interacted
      liked_ids = set(self.likes[self.likes['liker_id'] == user_id]['liked_id'])
      matched_ids = set(
          self.matches[self.matches['user1_id'] == user_id]['user2_id'].tolist() +
          self.matches[self.matches['user2_id'] == user_id]['user1_id'].tolist()
      )
      excluded_ids = liked_ids | matched_ids | {user_id}

      valid_indices = valid_indices[
          ~self.profiles.iloc[valid_indices]['id'].isin(excluded_ids)
      ]

      # Nếu hết candidate thì trả empty
      if len(valid_indices) == 0:
          return pd.DataFrame(columns=['id', 'username', 'gender', 'age', 'city', 'hobby_count', 'content_score'])

      # Get top similar
      valid_similarities = [(idx, float(similarities[idx])) for idx in valid_indices]
      valid_similarities.sort(key=lambda x: x[1], reverse=True)

      top_pairs = valid_similarities[:top_n]
      top_indices = [idx for idx, _ in top_pairs]

      recommendations = self.profiles.iloc[top_indices].copy()
      recommendations['content_score'] = [sim for _, sim in top_pairs]

      # ✅ Return age_raw nhưng rename thành age để hiển thị đúng
      return (
          recommendations[['id', 'username', 'gender', 'age_raw', 'city', 'hobby_count', 'content_score']]
          .rename(columns={'age_raw': 'age'})
          .reset_index(drop=True)
      )

    def get_collaborative_recommendations(self, user_id, top_n=10):
      if user_id not in self.user_id_to_idx:
          print(f"❌ User {user_id} không có interaction data!")
          return pd.DataFrame()

      colmap = self._get_profile_cols()
      username_col = colmap['username']

      # lấy user_row để filter pref
      user_row = self.profiles[self.profiles['id'] == user_id].iloc[0]

      user_idx = self.user_id_to_idx[user_id]
      user_similarities = self.user_similarity_matrix[user_idx]

      predicted_scores = np.zeros(len(self.user_id_to_idx))

      for profile_idx in range(len(self.user_id_to_idx)):
          if profile_idx == user_idx:
              continue
          liked_by = self.interaction_matrix[:, profile_idx]
          if liked_by.sum() > 0:
              predicted_scores[profile_idx] = np.dot(user_similarities, liked_by) / (user_similarities.sum() + 1e-10) #công thức Collaborative Filtering

      profile_ids = [self.idx_to_user_id[idx] for idx in range(len(self.user_id_to_idx))]

      # exclude interacted
      liked_ids = set(self.likes[self.likes['liker_id'] == user_id]['liked_id'])
      matched_ids = set(
          self.matches[self.matches['user1_id'] == user_id]['user2_id'].tolist() +
          self.matches[self.matches['user2_id'] == user_id]['user1_id'].tolist()
      )
      excluded_ids = liked_ids | matched_ids | {user_id}

      candidates = self.profiles[self.profiles['id'].isin(profile_ids)].copy()
      candidates = candidates[~candidates['id'].isin(excluded_ids)]

      # ✅ apply pref_gender
      if pd.notna(user_row.get('pref_gender', np.nan)) and user_row['pref_gender'] != 'BOTH':
          candidates = candidates[candidates['gender'] == user_row['pref_gender']]

      # ✅ apply pref ages (dùng age_raw)
      if 'age_raw' in candidates.columns:
          if pd.notna(user_row.get('pref_min_age', np.nan)):
              candidates = candidates[candidates['age_raw'] >= user_row['pref_min_age']]
          if pd.notna(user_row.get('pref_max_age', np.nan)):
              candidates = candidates[candidates['age_raw'] <= user_row['pref_max_age']]

      if len(candidates) == 0:
          return pd.DataFrame()

      # pick top by predicted_scores
      id_to_score = {pid: predicted_scores[self.user_id_to_idx[pid]] for pid in candidates['id'].values}
      candidates['collab_score'] = candidates['id'].map(id_to_score).fillna(0.0)
      candidates = candidates.sort_values('collab_score', ascending=False).head(top_n)

      # chuẩn hoá cột trả về (rename username nếu dataset dùng name)
      candidates['username'] = candidates[username_col]

      out = candidates[['id', 'username', 'gender', 'age_raw', 'city', 'collab_score']].copy()
      out = out.rename(columns={'age_raw': 'age'})
      return out.reset_index(drop=True)


    def get_hybrid_recommendations(self, user_id, top_n=10, content_weight=0.6, collab_weight=0.4):
      content_recs = self.get_content_recommendations(user_id, top_n=top_n*3) #Chuyển scores về range [0, 1]
      collab_recs = self.get_collaborative_recommendations(user_id, top_n=top_n*3)

      if content_recs.empty and collab_recs.empty:
          print(f"❌ Không thể tạo recommendations cho user {user_id}")
          return pd.DataFrame()

      # normalize scores
      if not content_recs.empty:
          min_val = content_recs['content_score'].min()
          max_val = content_recs['content_score'].max()
          content_recs['norm_content'] = (content_recs['content_score'] - min_val) / (max_val - min_val + 1e-10)

      if not collab_recs.empty:
          min_val = collab_recs['collab_score'].min()
          max_val = collab_recs['collab_score'].max()
          collab_recs['norm_collab'] = (collab_recs['collab_score'] - min_val) / (max_val - min_val + 1e-10)

      if content_recs.empty:
          hybrid = collab_recs[['id', 'norm_collab']].copy()
          hybrid['norm_content'] = 0.0
      elif collab_recs.empty:
          hybrid = content_recs[['id', 'norm_content']].copy()
          hybrid['norm_collab'] = 0.0
      else:
          hybrid = content_recs[['id', 'norm_content']].merge(
              collab_recs[['id', 'norm_collab']],
              on='id',
              how='outer'
          ).fillna(0.0)

      # ✅ pref_gender “đánh mạnh hơn”: Đảm bảo recommend đúng giới tính user muốn
      # Nếu user có pref_gender cụ thể => tăng content_weight (ưu tiên match giới tính)
      user_row = self.profiles[self.profiles['id'] == user_id].iloc[0]
      if pd.notna(user_row.get('pref_gender', np.nan)) and user_row['pref_gender'] != 'BOTH':
          content_weight = max(content_weight, 0.75)
          collab_weight = 1.0 - content_weight

      hybrid['hybrid_score'] = content_weight * hybrid['norm_content'] + collab_weight * hybrid['norm_collab']
      hybrid = hybrid.sort_values('hybrid_score', ascending=False).head(top_n)

      # Merge thông tin profile (robust theo dataset)
      colmap = self._get_profile_cols()
      username_col = colmap['username']

      info_cols = ['id', 'gender', 'city']
      if 'age_raw' in self.profiles.columns:
          info_cols.append('age_raw')
      elif 'age' in self.profiles.columns:
          info_cols.append('age')

      info_cols.append(username_col)

      info = self.profiles[info_cols].copy()
      info = info.rename(columns={username_col: 'username'})
      if 'age_raw' in info.columns:
          info = info.rename(columns={'age_raw': 'age'})

      result = hybrid.merge(info, on='id', how='left')

      # đảm bảo không trả nhầm giới tính (filter lại 1 lần cuối)
      if pd.notna(user_row.get('pref_gender', np.nan)) and user_row['pref_gender'] != 'BOTH':
          result = result[result['gender'] == user_row['pref_gender']]

      return result[['id', 'username', 'gender', 'age', 'city', 'hybrid_score']].reset_index(drop=True)


    def _get_profile_cols(self):
      """
      Chuẩn hoá tên cột để tránh KeyError khi dataset dùng tên khác nhau
      """
      cols = set(self.profiles.columns)

      # candidates theo các tên hay gặp
      username_col = 'username' if 'username' in cols else ('name' if 'name' in cols else None)
      city_col = 'city' if 'city' in cols else ('location_name' if 'location_name' in cols else None)

      # bắt buộc tối thiểu
      if username_col is None:
          raise KeyError("Không tìm thấy cột tên user: cần 'username' hoặc 'name'")
      if city_col is None:
          # vẫn cho chạy, nhưng city sẽ là Unknown
          city_col = None

      return {
          'username': username_col,
          'city': city_col
      }

    def print_summary(self):
        """In thống kê hệ thống"""
        print("\n" + "="*70)
        print("THỐNG KÊ HỆ THỐNG RECOMMENDATION")
        print("="*70)
        print(f"\n📊 Dữ liệu:")
        print(f"   - Profiles: {len(self.profiles)}")
        print(f"   - Likes: {len(self.likes)}")
        print(f"   - Matches: {len(self.matches)}")
        print(f"\n🎯 Features:")
        print(f"   - Content features: {self.content_feature_matrix.shape[1]}")
        print(f"   - Interaction matrix: {self.interaction_matrix.shape}")
        print(f"\n✨ Engagement:")
        print(f"   - Avg likes given: {self.profiles['likes_given_count'].mean():.1f}")
        print(f"   - Avg match rate: {self.profiles['match_rate'].mean()*100:.1f}%")
        print("="*70)


# Đánh giá mô hình

In [90]:
def precision_at_k(recommendations, actual_liked_ids, k=10):
    """
    Tính Precision@K - Trong top K recommendations, bao nhiêu % thực sự được like

    Args:
        recommendations: List hoặc array các recommended user_ids
        actual_liked_ids: Set các user_ids thực sự được like
        k: Số lượng top recommendations để đánh giá

    Returns:
        float: Precision@K score (0-1)
    """
    if len(recommendations) == 0:
        return 0.0

    top_k = recommendations[:k]
    hits = len(set(top_k) & set(actual_liked_ids))
    return hits / k


In [91]:
def recall_at_k(recommendations, actual_liked_ids, k=10):
    """
    Tính Recall@K - Trong tất cả người được like, tìm được bao nhiêu % trong top K

    Args:
        recommendations: List hoặc array các recommended user_ids
        actual_liked_ids: Set các user_ids thực sự được like
        k: Số lượng top recommendations để đánh giá

    Returns:
        float: Recall@K score (0-1)
    """
    if len(actual_liked_ids) == 0:
        return 0.0

    top_k = recommendations[:k]
    hits = len(set(top_k) & set(actual_liked_ids))
    return hits / len(actual_liked_ids)

In [92]:
def average_precision(recommendations, actual_liked_ids):
    """
    Tính Average Precision (AP) - Xét đến vị trí của các hits trong recommendations

    Args:
        recommendations: List các recommended user_ids (đã sắp xếp theo score)
        actual_liked_ids: Set các user_ids thực sự được like

    Returns:
        float: Average Precision score (0-1)
    """
    if len(actual_liked_ids) == 0:
        return 0.0

    hits = 0
    sum_precisions = 0.0

    for i, rec_id in enumerate(recommendations, 1):
        if rec_id in actual_liked_ids:
            hits += 1
            precision_at_i = hits / i
            sum_precisions += precision_at_i

    return sum_precisions / len(actual_liked_ids) if hits > 0 else 0.0

In [93]:
def mean_average_precision(all_recommendations, all_actual_likes):
    """
    Tính MAP (Mean Average Precision) - Trung bình AP của tất cả users

    Args:
        all_recommendations: Dict {user_id: [recommended_ids]}
        all_actual_likes: Dict {user_id: [actual_liked_ids]}

    Returns:
        float: MAP score (0-1)
    """
    aps = []
    for user_id in all_recommendations:
        if user_id in all_actual_likes:
            ap = average_precision(
                all_recommendations[user_id],
                all_actual_likes[user_id]
            )
            aps.append(ap)

    return np.mean(aps) if aps else 0.0

In [94]:
def ndcg_at_k(recommendations, actual_liked_ids, k=10):
    """
    Tính NDCG@K (Normalized Discounted Cumulative Gain)
    Xét đến thứ tự: hit ở vị trí cao được tính điểm cao hơn

    Args:
        recommendations: List các recommended user_ids
        actual_liked_ids: Set các user_ids thực sự được like
        k: Số lượng top recommendations

    Returns:
        float: NDCG@K score (0-1)
    """
    if len(actual_liked_ids) == 0:
        return 0.0

    top_k = recommendations[:k]

    # DCG - Discounted Cumulative Gain
    dcg = 0.0
    for i, rec_id in enumerate(top_k, 1):
        if rec_id in actual_liked_ids:
            dcg += 1.0 / np.log2(i + 1)

    # IDCG - Ideal DCG (nếu tất cả hits ở top)
    idcg = 0.0
    for i in range(1, min(len(actual_liked_ids), k) + 1):
        idcg += 1.0 / np.log2(i + 1)

    return dcg / idcg if idcg > 0 else 0.0

In [95]:
def hit_rate_at_k(recommendations, actual_liked_ids, k=10):
    """
    Tính Hit Rate@K - Có ít nhất 1 hit trong top K không?

    Args:
        recommendations: List các recommended user_ids
        actual_liked_ids: Set các user_ids thực sự được like
        k: Số lượng top recommendations

    Returns:
        int: 1 nếu có hit, 0 nếu không
    """
    top_k = recommendations[:k]
    return 1 if len(set(top_k) & set(actual_liked_ids)) > 0 else 0

In [96]:
def coverage(all_recommendations, total_items):
    """
    Tính Coverage - Tỷ lệ items được recommend ít nhất 1 lần

    Args:
        all_recommendations: Dict {user_id: [recommended_ids]}
        total_items: Tổng số items (profiles) trong hệ thống

    Returns:
        float: Coverage score (0-1)
    """
    recommended_items = set()
    for recs in all_recommendations.values():
        recommended_items.update(recs)

    return len(recommended_items) / total_items if total_items > 0 else 0.0

In [97]:
def diversity_score(recommendations, profiles_df):
    """
    Tính Diversity - Độ đa dạng của recommendations
    Dựa trên sự khác biệt về features (age, city, hobbies)

    Args:
        recommendations: List các recommended user_ids
        profiles_df: DataFrame profiles với features

    Returns:
        float: Diversity score (0-1), càng cao càng đa dạng
    """
    if len(recommendations) <= 1:
        return 0.0

    rec_profiles = profiles_df[profiles_df['id'].isin(recommendations)]

    if len(rec_profiles) <= 1:
        return 0.0

    # Tính độ đa dạng về tuổi
    age_std = rec_profiles['age'].std() if 'age' in rec_profiles else 0
    age_diversity = min(age_std / 10, 1.0)  # Normalize

    # Tính độ đa dạng về location
    city_unique = rec_profiles['city'].nunique() if 'city' in rec_profiles else 0
    city_diversity = city_unique / len(rec_profiles)

    # Tính độ đa dạng về hobby
    hobby_diversity = 0.0
    if 'hobby_count' in rec_profiles.columns:
        hobby_std = rec_profiles['hobby_count'].std()
        hobby_diversity = min(hobby_std / 5, 1.0)

    # Average diversity
    return (age_diversity + city_diversity + hobby_diversity) / 3

In [98]:
class RecommendationEvaluator:
    """
    Class để đánh giá toàn diện hệ thống recommendation
    """

    def __init__(self, rcm_system, test_likes_df):
        """
        Khởi tạo evaluator

        Args:
            rcm_system: DatingRecommendationSystem đã train
            test_likes_df: DataFrame likes để test (future likes)
        """
        self.rcm_system = rcm_system
        self.test_likes = test_likes_df

    def train_test_split_temporal(self, likes_df, test_ratio=0.2):
      likes_df = likes_df.copy()

      # ✅ Parse timestamp robust (có/không có microseconds đều được)
      likes_df['timestamp'] = pd.to_datetime(
          likes_df['timestamp'],
          format='mixed',      # pandas tự nhận từng dòng
          errors='coerce',     # dòng lỗi -> NaT
          utc=True             # nếu muốn đồng nhất timezone
      )

      # Nếu có dòng lỗi parse -> loại bỏ để tránh crash
      bad = likes_df['timestamp'].isna().sum()
      if bad > 0:
          print(f"⚠️ Có {bad} timestamp parse lỗi -> drop")
          likes_df = likes_df.dropna(subset=['timestamp'])

      likes_df = likes_df.sort_values('timestamp')

      split_idx = int(len(likes_df) * (1 - test_ratio))
      train_df = likes_df.iloc[:split_idx].copy()
      test_df = likes_df.iloc[split_idx:].copy()

      print(f"📊 Train/Test Split:")
      print(f"   - Train: {len(train_df)} likes ({(1-test_ratio)*100:.0f}%)")
      print(f"   - Test: {len(test_df)} likes ({test_ratio*100:.0f}%)")

      return train_df, test_df

    def evaluate_single_user(self, user_id, k=10, method='hybrid'):
        """
        Đánh giá recommendations cho 1 user cụ thể

        Args:
            user_id: ID của user
            k: Top K recommendations
            method: 'content', 'collaborative', hoặc 'hybrid'

        Returns:
            dict: Các metrics
        """
        # Get recommendations
        if method == 'content':
            recs_df = self.rcm_system.get_content_recommendations(user_id, top_n=k)
        elif method == 'collaborative':
            recs_df = self.rcm_system.get_collaborative_recommendations(user_id, top_n=k)
        else:  # hybrid
            recs_df = self.rcm_system.get_hybrid_recommendations(user_id, top_n=k)

        if recs_df.empty:
            return None

        recommendations = recs_df['id'].tolist()

        # Get actual likes in test set
        actual_likes = self.test_likes[
            self.test_likes['liker_id'] == user_id
        ]['liked_id'].tolist()

        if not actual_likes:
            return None

        # Calculate metrics
        metrics = {
            'user_id': user_id,
            'precision@k': precision_at_k(recommendations, actual_likes, k),
            'recall@k': recall_at_k(recommendations, actual_likes, k),
            'ndcg@k': ndcg_at_k(recommendations, actual_likes, k),
            'hit_rate@k': hit_rate_at_k(recommendations, actual_likes, k),
            'diversity': diversity_score(recommendations, self.rcm_system.profiles)
        }

        return metrics

    def evaluate_all_users(self, k=10, method='hybrid', sample_size=None):
        """
        Đánh giá recommendations cho tất cả users

        Args:
            k: Top K recommendations
            method: 'content', 'collaborative', hoặc 'hybrid'
            sample_size: Số user để sample (None = tất cả)

        Returns:
            DataFrame với metrics cho từng user
        """
        print(f"\n{'='*70}")
        print(f"ĐÁNH GIÁ HỆ THỐNG - METHOD: {method.upper()}")
        print(f"{'='*70}\n")

        # Get users có likes trong test set
        test_users = self.test_likes['liker_id'].unique()

        if sample_size:
            test_users = np.random.choice(
                test_users,
                min(sample_size, len(test_users)),
                replace=False
            )

        print(f"🔍 Đánh giá {len(test_users)} users...")

        results = []
        for i, user_id in enumerate(test_users, 1):
            if i % 10 == 0:
                print(f"   Progress: {i}/{len(test_users)}")

            metrics = self.evaluate_single_user(user_id, k, method)
            if metrics:
                results.append(metrics)

        if not results:
            print("❌ Không có kết quả đánh giá!")
            return pd.DataFrame()

        results_df = pd.DataFrame(results)

        # Print summary
        print(f"\n{'='*70}")
        print(f"KẾT QUẢ ĐÁNH GIÁ (K={k})")
        print(f"{'='*70}")
        print(f"\n📊 Average Metrics:")
        print(f"   - Precision@{k}:  {results_df['precision@k'].mean():.4f}")
        print(f"   - Recall@{k}:     {results_df['recall@k'].mean():.4f}")
        print(f"   - NDCG@{k}:       {results_df['ndcg@k'].mean():.4f}")
        print(f"   - Hit Rate@{k}:   {results_df['hit_rate@k'].mean():.4f}")
        print(f"   - Diversity:      {results_df['diversity'].mean():.4f}")
        print(f"\n📈 Distribution:")
        print(results_df.describe())
        print(f"{'='*70}\n")

        return results_df

    def compare_methods(self, k=10, sample_size=None):
        """
        So sánh 3 methods: content-based, collaborative, hybrid

        Args:
            k: Top K recommendations
            sample_size: Số user để sample

        Returns:
            DataFrame so sánh các methods
        """
        print(f"\n{'='*70}")
        print("SO SÁNH CÁC PHƯƠNG PHÁP RECOMMENDATION")
        print(f"{'='*70}\n")

        methods = ['content', 'collaborative', 'hybrid']
        comparison = []

        for method in methods:
            print(f"\n🔄 Đánh giá {method.upper()}...")
            results_df = self.evaluate_all_users(k, method, sample_size)

            if not results_df.empty:
                comparison.append({
                    'method': method,
                    'precision@k': results_df['precision@k'].mean(),
                    'recall@k': results_df['recall@k'].mean(),
                    'ndcg@k': results_df['ndcg@k'].mean(),
                    'hit_rate@k': results_df['hit_rate@k'].mean(),
                    'diversity': results_df['diversity'].mean()
                })

        comparison_df = pd.DataFrame(comparison)

        print(f"\n{'='*70}")
        print("KẾT QUẢ SO SÁNH")
        print(f"{'='*70}")
        print(comparison_df.to_string(index=False))
        print(f"{'='*70}\n")

        # Tìm method tốt nhất cho từng metric
        print("🏆 BEST METHODS:")
        for col in comparison_df.columns:
            if col != 'method':
                best_method = comparison_df.loc[comparison_df[col].idxmax(), 'method']
                best_score = comparison_df[col].max()
                print(f"   - {col}: {best_method.upper()} ({best_score:.4f})")

        return comparison_df

# HELPER FUNCTIONS


In [99]:
def display_recommendations(recommendations, title="RECOMMENDATIONS"):
    """
    Hiển thị recommendations đẹp mắt

    Args:
        recommendations: DataFrame recommendations
        title: Tiêu đề
    """
    if recommendations.empty:
        print("❌ Không có recommendations")
        return

    print("\n" + "="*70)
    print(f"🎯 {title}")
    print("="*70)
    print(recommendations.to_string(index=False))
    print("="*70)


# Quá trình xây dựng bộ dữ liệu giả lập cho user

In [100]:
fake_profiles = generate_fake_profiles(n_profiles=1500, base_profiles_df=real_profiles)

✅ Đã tạo 1500 fake profiles
   - Nam: 808 (53.9%)
   - Nữ: 692 (46.1%)
   - Tuổi trung bình: 23.5


In [101]:
all_profiles = combine_real_and_fake_data(real_profiles, fake_profiles)


KẾT HỢP REAL & FAKE DATA

✅ Kết hợp thành công!
   - Real data: 17 rows
   - Fake data: 1500 rows
   - Total: 1517 rows


In [102]:
all_profiles.to_csv("all_profiles.csv", index=False, encoding="utf-8-sig")

In [103]:
fake_likes = generate_fake_likes(all_profiles, n_likes=50000)



TẠO 50000 FAKE LIKES

✅ Đã tạo 48495 fake likes
   - Unique likers: 1507
   - Unique liked: 1517
   - Avg likes per user: 32.2


In [104]:
all_likes = combine_real_and_fake_data(real_likes, fake_likes)


KẾT HỢP REAL & FAKE DATA

✅ Kết hợp thành công!
   - Real data: 75 rows
   - Fake data: 48495 rows
   - Total: 48570 rows


In [105]:
all_likes.to_csv("all_likes.csv", index=False, encoding="utf-8-sig")

In [106]:
fake_matches = generate_fake_matches(all_likes, match_rate=0.4)


TẠO FAKE MATCHES (Match rate: 40%)

✅ Đã tạo 581 fake matches
   - Unique users in matches: 806
   - Actual match rate: 1.20%


In [107]:
all_matches = combine_real_and_fake_data(real_matches, fake_matches)


KẾT HỢP REAL & FAKE DATA

✅ Kết hợp thành công!
   - Real data: 11 rows
   - Fake data: 581 rows
   - Total: 592 rows


In [108]:
all_matches.to_csv("all_matches.csv", index=False, encoding="utf-8-sig")

# FEATURE ENGINEERING/

In [109]:
enriched_profiles = enrich_all_features(all_profiles, all_likes, all_matches)

BẮT ĐẦU FEATURE ENGINEERING
🔄 Đang tính tuổi từ date_of_birth...
✅ Đã tính tuổi cho 1517 profiles
   - Tuổi trung bình: 23.5
   - Tuổi min-max: 18-37

🔄 Đang parse hobby features...
✅ Đã parse hobby cho 1517 profiles
   - Số hobby trung bình: 3.6
   - Max hobbies: 5

🔄 Đang tạo bio features...
✅ Đã tạo bio features
   - % có bio: 99.9%
   - Độ dài bio trung bình: 27 ký tự

🔄 Đang tính tuổi tài khoản...
✅ Đã tính tuổi tài khoản
   - Tuổi tài khoản trung bình: 51 ngày
   - Tuổi tài khoản min-max: 1-116 ngày

🔄 Đang tính profile completeness...
✅ Đã tính profile completeness
   - Completeness trung bình: 71.6%

🔄 Đang tính engagement metrics...
✅ Đã tính engagement metrics
   - Avg likes given: 32.0
   - Avg likes received: 32.0
   - Avg matches: 0.8
   - Avg match rate: 2.5%

🔄 Đang chuẩn hóa location...
✅ Đã chuẩn hóa location
   - Số thành phố unique: 33

HOÀN THÀNH FEATURE ENGINEERING

📊 Thống kê:
   - Tổng số profiles: 1517
   - Tổng số features: 31

✨ Các features mới đã tạo:
    1.

In [110]:
save_enriched_data(enriched_profiles, "/content/enriched_profiles_fullSS.csv")


💾 Đang lưu dữ liệu vào /content/enriched_profiles_fullSS.csv...
✅ Đã lưu thành công!


'/content/enriched_profiles_fullSS.csv'

In [111]:
evaluator_temp = RecommendationEvaluator(None, None)
train_likes, test_likes = evaluator_temp.train_test_split_temporal(all_likes, test_ratio=0.2)

📊 Train/Test Split:
   - Train: 38856 likes (80%)
   - Test: 9714 likes (20%)


In [112]:
rcm_system = DatingRecommendationSystem(enriched_profiles, train_likes, all_matches)
rcm_system.print_summary()


KHỞI TẠO RECOMMENDATION SYSTEM

🔄 Đang chuẩn bị Content-based features...
✅ Content feature matrix: (1517, 31)

🔄 Đang chuẩn bị Collaborative features...
✅ Interaction matrix: (1517, 1517)
✅ Tổng interactions: 38855

✅ Khởi tạo thành công!

THỐNG KÊ HỆ THỐNG RECOMMENDATION

📊 Dữ liệu:
   - Profiles: 1517
   - Likes: 38856
   - Matches: 592

🎯 Features:
   - Content features: 31
   - Interaction matrix: (1517, 1517)

✨ Engagement:
   - Avg likes given: 32.0
   - Avg match rate: 2.5%


In [113]:
user_id = 1  # Hoặc random user
print(f"\n{'='*70}")
print(f"TEST RECOMMENDATIONS CHO USER {user_id}")
print(f"{'='*70}")



TEST RECOMMENDATIONS CHO USER 1


In [114]:
content_recs = rcm_system.get_content_recommendations(user_id, top_n=10)
display_recommendations(content_recs, "CONTENT-BASED")


🎯 CONTENT-BASED
  id    username gender  age                       city  hobby_count  content_score
 938      Trí Hà   MALE   25   Phường Ô Chợ Dừa, Hà Nội            5       0.802607
 281    Huy Thái   MALE   24   Phường Ô Chợ Dừa, Hà Nội            4       0.797058
 717   Kiên Trần   MALE   21             Quận 1, TP.HCM            5       0.779321
 786     Long Tạ   MALE   20 Phường Khương Đình, Hà Nội            5       0.776283
 971 Hoàng Huỳnh   MALE   23 Phường Khương Đình, Hà Nội            5       0.775361
1394   Luân Đặng   MALE   18   Phường Ô Chợ Dừa, Hà Nội            3       0.770973
 653    Sơn Thái   MALE   23 Phường Khương Đình, Hà Nội            2       0.761922
 743   Anh Quách   MALE   23 Phường Khương Đình, Hà Nội            5       0.760394
  15 Quang Thắng   MALE   22 Phường Khương Đình, Hà Nội            5       0.758061
1314  Thịnh Đặng   MALE   24             Quận 3, TP.HCM            5       0.754990


In [115]:
collab_recs = rcm_system.get_collaborative_recommendations(user_id, top_n=10)
display_recommendations(collab_recs, "COLLABORATIVE")



🎯 COLLABORATIVE
  id    username gender  age                       city  collab_score
  14   Nhật Minh   MALE   25                    Unknown      0.057512
1048   Hạnh Tăng FEMALE   27     TP Huế, Thừa Thiên Huế      0.046034
  18  Phát Dương   MALE   25     Quận Hoàn Kiếm, Hà Nội      0.045872
 478    Tuyết Lê FEMALE   25      Quận Tân Bình, TP.HCM      0.042493
 311 Hồng Nguyễn FEMALE   28       Quận Đống Đa, Hà Nội      0.039647
1232     Minh Đỗ   MALE   28 Phường Khương Đình, Hà Nội      0.039096
1443     Đạt Cao   MALE   26    TP Nha Trang, Khánh Hòa      0.038189
  64   Oanh Trần FEMALE   27   Phường Ô Chợ Dừa, Hà Nội      0.036812
 445      Sơn Vũ   MALE   23     Quận Long Biên, Hà Nội      0.036573
 332   Bình Đinh   MALE   26 TP Thủ Dầu Một, Bình Dương      0.036362


In [116]:
hybrid_recs = rcm_system.get_hybrid_recommendations(user_id, top_n=10)
display_recommendations(hybrid_recs, "HYBRID")


🎯 HYBRID
  id    username gender  age                       city  hybrid_score
 938      Trí Hà   MALE   25   Phường Ô Chợ Dừa, Hà Nội      0.600000
 281    Huy Thái   MALE   24   Phường Ô Chợ Dừa, Hà Nội      0.563878
 717   Kiên Trần   MALE   21             Quận 1, TP.HCM      0.448421
 786     Long Tạ   MALE   20 Phường Khương Đình, Hà Nội      0.428638
 971 Hoàng Huỳnh   MALE   23 Phường Khương Đình, Hà Nội      0.422643
  14   Nhật Minh   MALE   25                    Unknown      0.400000
1394   Luân Đặng   MALE   18   Phường Ô Chợ Dừa, Hà Nội      0.394075
 653    Sơn Thái   MALE   23 Phường Khương Đình, Hà Nội      0.335158
 743   Anh Quách   MALE   23 Phường Khương Đình, Hà Nội      0.325208
  15 Quang Thắng   MALE   22 Phường Khương Đình, Hà Nội      0.310021


In [117]:
evaluator = RecommendationEvaluator(rcm_system, test_likes)

In [118]:
print("\n🔬 EVALUATING MODELS...")
comparison = evaluator.compare_methods(k=10, sample_size=100)


🔬 EVALUATING MODELS...

SO SÁNH CÁC PHƯƠNG PHÁP RECOMMENDATION


🔄 Đánh giá CONTENT...

ĐÁNH GIÁ HỆ THỐNG - METHOD: CONTENT

🔍 Đánh giá 100 users...
   Progress: 10/100
   Progress: 20/100
   Progress: 30/100
   Progress: 40/100
   Progress: 50/100
   Progress: 60/100
   Progress: 70/100
   Progress: 80/100
   Progress: 90/100
   Progress: 100/100

KẾT QUẢ ĐÁNH GIÁ (K=10)

📊 Average Metrics:
   - Precision@10:  0.0110
   - Recall@10:     0.0160
   - NDCG@10:       0.0113
   - Hit Rate@10:   0.0900
   - Diversity:      0.3396

📈 Distribution:
           user_id  precision@k    recall@k      ndcg@k  hit_rate@k  \
count   100.000000   100.000000  100.000000  100.000000  100.000000   
mean    800.680000     0.011000    0.015956    0.011258    0.090000   
std     449.926882     0.037322    0.058014    0.040303    0.287623   
min       1.000000     0.000000    0.000000    0.000000    0.000000   
25%     394.500000     0.000000    0.000000    0.000000    0.000000   
50%     827.500000     0.

In [119]:
print("\n📈 TESTING DIFFERENT K VALUES...")
k_results = []
for k in [5, 10, 20, 30]:
    results = evaluator.evaluate_all_users(k=k, method='hybrid', sample_size=50)
    if not results.empty:
        k_results.append({
            'K': k,
            'Precision': results['precision@k'].mean(),
            'Recall': results['recall@k'].mean(),
            'NDCG': results['ndcg@k'].mean()
        })

k_df = pd.DataFrame(k_results)
print("\n" + "="*70)
print("KẾT QUẢ THEO K:")
print("="*70)
print(k_df.to_string(index=False))


📈 TESTING DIFFERENT K VALUES...

ĐÁNH GIÁ HỆ THỐNG - METHOD: HYBRID

🔍 Đánh giá 50 users...
   Progress: 10/50
   Progress: 20/50
   Progress: 30/50
   Progress: 40/50
   Progress: 50/50

KẾT QUẢ ĐÁNH GIÁ (K=5)

📊 Average Metrics:
   - Precision@5:  0.0080
   - Recall@5:     0.0118
   - NDCG@5:       0.0115
   - Hit Rate@5:   0.0400
   - Diversity:      0.3721

📈 Distribution:
           user_id  precision@k   recall@k     ndcg@k  hit_rate@k  diversity
count    50.000000     50.00000  50.000000  50.000000   50.000000  50.000000
mean    683.580000      0.00800   0.011818   0.011527    0.040000   0.372144
std     419.435923      0.03959   0.071611   0.057967    0.197949   0.064250
min      39.000000      0.00000   0.000000   0.000000    0.000000   0.230401
25%     334.000000      0.00000   0.000000   0.000000    0.000000   0.324260
50%     702.000000      0.00000   0.000000   0.000000    0.000000   0.374025
75%     916.250000      0.00000   0.000000   0.000000    0.000000   0.409801
max

# Xây dựng mô hình với lightFM

In [120]:
!apt-get -y update|

/bin/bash: -c: line 2: syntax error: unexpected end of file


In [121]:
!apt-get -y install build-essential python3-dev

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
build-essential is already the newest version (12.9ubuntu3).
The following additional packages will be installed:
  javascript-common libjs-sphinxdoc libjs-underscore python3.10-dev
Suggested packages:
  apache2 | lighttpd | httpd
The following NEW packages will be installed:
  javascript-common libjs-sphinxdoc libjs-underscore python3-dev
  python3.10-dev
0 upgraded, 5 newly installed, 0 to remove and 2 not upgraded.
Need to get 796 kB of archives.
After this operation, 1,257 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 javascript-common all 11+nmu1 [5,936 B]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libjs-underscore all 1.13.2~dfsg-2 [118 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 libjs-sphinxdoc all 4.3.2-1 [139 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 python3.10-dev amd64 3.10.

In [122]:
!pip -q install --upgrade pip setuptools wheel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 61.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


In [123]:
!pip -q install --upgrade pip

In [124]:
!pip -q install lightfm-next

In [125]:
from lightfm import LightFM
from lightfm.data import Dataset
from lightfm.evaluation import precision_at_k, recall_at_k, auc_score
from scipy import sparse
import numpy as np
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
import ast

In [126]:
class LightFMDatingRecommender:
    """
    LightFM-based recommendation system cho dating app
    Sử dụng hybrid approach với user và item features
    """

    def __init__(self, profiles_df, likes_df, matches_df):
        """
        Khởi tạo LightFM recommender

        Args:
            profiles_df: DataFrame profiles đã được làm giàu
            likes_df: DataFrame likes
            matches_df: DataFrame matches
        """
        print("\n" + "="*70)
        print("KHỞI TẠO LIGHTFM RECOMMENDATION SYSTEM")
        print("="*70)

        self.profiles = profiles_df.copy()
        self.likes = likes_df.copy()
        self.matches = matches_df.copy()

        # Đảm bảo có age_raw (lấy năm hiện tại trừ đi cột năm sinh)
        if 'age_raw' not in self.profiles.columns:
            if 'age' in self.profiles.columns:
                self.profiles['age_raw'] = self.profiles['age']
            else:
                self.profiles['age_raw'] = 2026 - pd.to_datetime(
                    self.profiles['date_of_birth']
                ).dt.year

        self.dataset = None
        self.model = None
        self.user_features_matrix = None
        self.item_features_matrix = None

        print("✅ Khởi tạo thành công!")

    def _parse_hobby_list(self, hobby_str):
        """Parse hobby từ string sang list"""
        try:
            if pd.isna(hobby_str):
                return []
            if isinstance(hobby_str, list):
                return hobby_str
            # Thử parse JSON string
            return ast.literal_eval(hobby_str)
        except:
            return []

    def prepare_dataset(self, train_likes):
      """
      Chuẩn bị dataset cho LightFM:
      - fit dataset mapping
      - build interactions + sample weights
      - build user/item features
      """
      print("\n📄 Chuẩn bị dataset cho LightFM...")

      # 1) Fit dataset (users + items)
      self.dataset = Dataset()

      user_ids = self.profiles["id"].unique().tolist()
      item_ids = self.profiles["id"].unique().tolist()  # dating: item cũng là user

      # build feature list
      user_feature_list = self._build_user_features()
      item_feature_list = self._build_item_features() if hasattr(self, "_build_item_features") else user_feature_list

      self.dataset.fit(
          users=user_ids,
          items=item_ids,
          user_features=user_feature_list,
          item_features=item_feature_list
      )

      # 2) Build interactions (+ weights nếu bạn có weight)
      # implicit: (user, item, weight)
      interactions_data = [
          (row["liker_id"], row["liked_id"], 1.0) for _, row in train_likes.iterrows()
      ]
      interactions, weights = self.dataset.build_interactions(interactions_data) #chuyển danh sách like thành ma trận thưa

      # 3) Build user features and item features
      user_feature_tuples = self._get_user_feature_tuples()
      self.user_features = self.dataset.build_user_features(user_feature_tuples, normalize=True) #tạo ma trận chứa các đặc điểm của người dùng

      # item features: vì item cũng là user, dùng chung tuple là hợp lý
      item_feature_tuples = self._get_item_feature_tuples() if hasattr(self, "_get_item_feature_tuples") else user_feature_tuples
      self.item_features = self.dataset.build_item_features(item_feature_tuples, normalize=True)

      # Lưu lại weights để train dùng được (nếu cần)
      self.weights = weights

      print("✅ Dataset chuẩn bị xong!")
      print(f"   - Interactions shape: {interactions.shape}")
      print(f"   - User features shape: {self.user_features.shape}")
      print(f"   - Item features shape: {self.item_features.shape}")
      print(f"   - Tổng interactions: {interactions.getnnz()}")
      self.train_interactions = interactions
      return interactions, weights


    def _build_user_features(self): #xây dựng thuộc tính (tạo các tag)
      """Xây dựng danh sách user features"""
      features = set()

      # Gender: tạo các tag giới tính (ví dụ: gender:1, gender:0).
      features.update([f"gender:{g}" for g in self.profiles['gender'].unique()])

      # Age groups - TÍNH TỪ DATA THỰC TẾ
      features.add("age_group:18")  # vì tuples có thể tạo ra 18 (từ 18-19)

      for i in range(20, 55, 5):    # 20,25,30,...,50
          features.add(f"age_group:{i}")

      # Target type
      features.update([
          f"target:{t}" for t in self.profiles['target_type'].unique()
          if pd.notna(t)
      ])

      # City
      features.update([
          f"city:{c}" for c in self.profiles['city'].unique()
          if pd.notna(c)
      ])

      # Hobbies
      all_hobbies = set()
      for hobby_list in self.profiles['hobby_list']:
          if isinstance(hobby_list, list):
              all_hobbies.update(hobby_list)
      features.update([f"hobby:{h}" for h in all_hobbies])

      # Profile completeness bins
      features.update([f"completeness:{i}" for i in range(0, 11)])

      return list(features)

    def _build_item_features(self):
        """Xây dựng danh sách item features (tương tự user)"""
        return self._build_user_features()

    def _get_user_feature_tuples(self):
        """Tạo tuples (user_id, [features]) cho mỗi user"""
        feature_tuples = []

        for _, row in self.profiles.iterrows():
            user_id = row['id']
            features = []

            # Gender
            if pd.notna(row['gender']):
                features.append(f"gender:{row['gender']}")

            # Age group
            age = row.get('age_raw', row.get('age', 25))
            if pd.notna(age):
                age_group = int(age // 5) * 5
                age_group = max(20, min(50, age_group))  # Giới hạn 18-50
                features.append(f"age_group:{age_group}")
            else:
                features.append("age_group:25")

            # Target type
            if pd.notna(row['target_type']):
                features.append(f"target:{row['target_type']}")

            # City
            if pd.notna(row['city']):
                features.append(f"city:{row['city']}")

            # Hobbies
            if isinstance(row['hobby_list'], list):
                for hobby in row['hobby_list']:
                    features.append(f"hobby:{hobby}")

            # Profile completeness
            comp = int(row.get('profile_completeness', 0) * 10)
            features.append(f"completeness:{comp}")

            feature_tuples.append((user_id, features)) #gom đúng thành định dạng cho Lightfm

        return feature_tuples

    def _get_item_feature_tuples(self):
        """Tạo tuples (item_id, [features]) cho mỗi item"""
        return self._get_user_feature_tuples()

    def train(self, interactions, epochs=30, num_threads=4, loss="warp", learning_rate=0.05, no_components=50, weights=None):
      print("\n🎓 Training LightFM model...")
      print(f"   - Loss: {loss}") #loss="bpr": Thuật toán tối ưu hóa việc xếp hạng (Bayesian Personalised Ranking).
      print(f"   - Epochs: {epochs}") #số lần lặp mà model train
      print(f"   - Components: {no_components}") #no_components=64: Số lượng chiều ẩn (latent factors). Máy tính sẽ tự tìm ra 64 đặc điểm ngầm định kết nối các người dùng với nhau.
      print(f"   - Learning rate: {learning_rate}")

      self.model = LightFM(
          loss=loss,
          learning_rate=learning_rate,
          no_components=no_components,
          random_state=42
      )

      # nếu không truyền weights vào thì lấy từ prepare_dataset()
      if weights is None:
          weights = getattr(self, "weights", None)

      # đảm bảo user/item features tồn tại
      if not hasattr(self, "user_features") or not hasattr(self, "item_features"):
          raise ValueError("Bạn cần gọi prepare_dataset() trước để tạo user_features và item_features.")

      self.model.fit(
          interactions,
          user_features=self.user_features,
          item_features=self.item_features,
          sample_weight=weights,
          epochs=epochs,
          num_threads=num_threads
      )

      print("✅ Train xong!")
      return self.model

    def predict_for_user(self, user_id, top_n=10, exclude_liked=True):
        """
        Dự đoán recommendations cho 1 user

        Args:
            user_id: ID của user
            top_n: Số lượng recommendations
            exclude_liked: Loại bỏ những người đã like

        Returns:
            DataFrame recommendations
        """
        if user_id not in self.dataset.mapping()[0]:
            print(f"⚠️ User {user_id} không có trong dataset")
            return pd.DataFrame()

        # Lấy user preferences để filter
        user_row = self.profiles[self.profiles['id'] == user_id].iloc[0]

        # Lấy all item IDs
        all_item_ids = list(self.dataset.mapping()[2].keys())

        # Filter candidates theo preferences
        candidates_df = self.profiles[self.profiles['id'].isin(all_item_ids)].copy()

        # Apply gender preference
        if pd.notna(user_row.get('pref_gender')) and user_row['pref_gender'] != 'BOTH':
            candidates_df = candidates_df[
                candidates_df['gender'] == user_row['pref_gender']
            ]

        # Apply age preference
        if pd.notna(user_row.get('pref_min_age')):
            candidates_df = candidates_df[
                candidates_df['age_raw'] >= user_row['pref_min_age']
            ]
        if pd.notna(user_row.get('pref_max_age')):
            candidates_df = candidates_df[
                candidates_df['age_raw'] <= user_row['pref_max_age']
            ]

        # Exclude self
        candidates_df = candidates_df[candidates_df['id'] != user_id]

        # Exclude liked/matched nếu cần
        if exclude_liked:
            liked_ids = set(self.likes[self.likes['liker_id'] == user_id]['liked_id'])
            matched_ids = set(
                self.matches[self.matches['user1_id'] == user_id]['user2_id'].tolist() +
                self.matches[self.matches['user2_id'] == user_id]['user1_id'].tolist()
            )
            excluded = liked_ids | matched_ids
            candidates_df = candidates_df[~candidates_df['id'].isin(excluded)]

        if len(candidates_df) == 0:
            return pd.DataFrame()

        candidate_ids = candidates_df['id'].tolist()

        # Convert IDs sang internal indices
        user_idx = self.dataset.mapping()[0][user_id]
        item_indices = [self.dataset.mapping()[2][iid] for iid in candidate_ids]

        # Predict scores
        scores = self.model.predict(
            user_idx,
            item_indices,
            user_features=self.user_features_matrix,
            item_features=self.item_features_matrix
        )

        # Tạo DataFrame với scores
        results = pd.DataFrame({
            'id': candidate_ids,
            'lightfm_score': scores
        })

        # Merge với profile info
        results = results.merge(
            self.profiles[['id', 'username', 'gender', 'age_raw', 'city']],
            on='id'
        )

        # Sort và lấy top N
        results = results.sort_values('lightfm_score', ascending=False).head(top_n)
        results = results.rename(columns={'age_raw': 'age'})

        return results.reset_index(drop=True)

    def evaluate(self, test_interactions, k=10, num_threads=4):
      print(f"\n📊 Đánh giá LightFM model (k={k})...")

      if not hasattr(self, "model") or self.model is None:
          raise ValueError("Model chưa được train. Hãy gọi train() trước.")

      if not hasattr(self, "train_interactions"):
          # fallback: nếu bạn chưa lưu, dùng test như train (không khuyến nghị)
          raise ValueError("Thiếu train_interactions. Hãy gọi prepare_dataset() và train() trước.")

      if not hasattr(self, "user_features") or not hasattr(self, "item_features"):
          raise ValueError("Thiếu user_features/item_features. Hãy gọi prepare_dataset() trước.")

      # ✅ luôn dùng features đúng như lúc train
      p = precision_at_k(
          self.model,
          test_interactions,
          train_interactions=self.train_interactions,
          user_features=self.user_features,
          item_features=self.item_features,
          k=k,
          num_threads=num_threads
      ).mean()

      r = recall_at_k(
          self.model,
          test_interactions,
          train_interactions=self.train_interactions,
          user_features=self.user_features,
          item_features=self.item_features,
          k=k,
          num_threads=num_threads
      ).mean()

      a = auc_score(
          self.model,
          test_interactions,
          train_interactions=self.train_interactions,
          user_features=self.user_features,
          item_features=self.item_features,
          num_threads=num_threads
      ).mean()

      metrics = {
          f"precision@{k}": float(p),
          f"recall@{k}": float(r),
          "auc": float(a)
      }

      print("✅ Kết quả đánh giá:")
      print(f"   - precision@{k}: {metrics[f'precision@{k}']:.4f}")
      print(f"   - recall@{k}: {metrics[f'recall@{k}']:.4f}")
      print(f"   - auc: {metrics['auc']:.4f}")

      return metrics

In [127]:

def compare_all_methods(content_based_system, lightfm_system, evaluator, k=10):
    """
    So sánh tất cả các methods: Content-based, Collaborative, Hybrid, và LightFM

    Args:
        content_based_system: DatingRecommendationSystem
        lightfm_system: LightFMDatingRecommender
        evaluator: RecommendationEvaluator
        k: Top K

    Returns:
        DataFrame so sánh
    """
    print("\n" + "="*70)
    print("SO SÁNH TẤT CẢ PHƯƠNG PHÁP RECOMMENDATION")
    print("="*70)

    # Evaluate traditional methods
    traditional_results = evaluator.compare_methods(k=k, sample_size=100)

    # Evaluate LightFM
    print("\nĐánh giá LightFM...")

    # Build test interactions
    test_interactions, _ = lightfm_system.dataset.build_interactions(
        [(row['liker_id'], row['liked_id'], 1.0)
         for _, row in evaluator.test_likes.iterrows()]
    )

    lightfm_metrics = lightfm_system.evaluate(test_interactions, k=k)

    # Add LightFM to comparison
    lightfm_row = pd.DataFrame([{
        'method': 'lightfm',
        'precision@k': lightfm_metrics[f'precision@{k}'],
        'recall@k': lightfm_metrics[f'recall@{k}'],
        'ndcg@k': 0.0,  # LightFM không tính NDCG mặc định
        'hit_rate@k': 0.0,
        'diversity': 0.0
    }])

    all_results = pd.concat([traditional_results, lightfm_row], ignore_index=True)

    print("\n" + "="*70)
    print("KẾT QUẢ SO SÁNH CUỐI CÙNG")
    print("="*70)
    print(all_results.to_string(index=False))
    print("="*70)

    # Find best method for each metric
    print("\n🏆 PHƯƠNG PHÁP TỐT NHẤT:")
    for col in ['precision@k', 'recall@k']:
        if col in all_results.columns:
            best_idx = all_results[col].idxmax()
            best_method = all_results.loc[best_idx, 'method']
            best_score = all_results.loc[best_idx, col]
            print(f"   - {col}: {best_method.upper()} ({best_score:.4f})")

    return all_results

In [129]:
enriched_profiles = pd.read_csv("/content/drive/MyDrive/Rcm-sys/data fake/fake/enriched_profiles_full.csv")

In [130]:
if 'age_raw' not in enriched_profiles.columns:
    if 'age' in enriched_profiles.columns:
        enriched_profiles['age_raw'] = enriched_profiles['age']
    else:
        enriched_profiles['age_raw'] = 2026 - pd.to_datetime(
            enriched_profiles['date_of_birth']
        ).dt.year

# Fix city nếu chưa có
if 'city' not in enriched_profiles.columns:
    enriched_profiles['city'] = enriched_profiles['location_name'].fillna('Unknown')

# Parse hobby_list nếu chưa có
if 'hobby_list' not in enriched_profiles.columns:
    import ast
    def parse_hobby(h):
        try:
            if pd.isna(h):
                return []
            if isinstance(h, list):
                return h
            return ast.literal_eval(h)
        except:
            return []
    enriched_profiles['hobby_list'] = enriched_profiles['hobby'].apply(parse_hobby)

In [131]:
# Parse hobby_list nếu chưa có
if 'hobby_list' not in enriched_profiles.columns:
    def parse_hobby(h):
        try:
            if pd.isna(h):
                return []
            if isinstance(h, list):
                return h
            return ast.literal_eval(h)
        except:
            return []

    enriched_profiles['hobby_list'] = enriched_profiles['hobby'].apply(parse_hobby)

# Đảm bảo có city column
if 'city' not in enriched_profiles.columns:
    enriched_profiles['city'] = enriched_profiles['location_name'].fillna('Unknown')
    print("✅ Đã tạo city column")

In [132]:
# Khởi tạo LightFM recommender
lightfm_rcm = LightFMDatingRecommender(
    enriched_profiles,
    train_likes,     # train set
    all_matches
)

# Chuẩn bị dataset
train_interactions, weights = lightfm_rcm.prepare_dataset(train_likes)

# Train model với các hyperparameters tối ưu
lightfm_rcm.train(
    train_interactions,
    epochs=40,
    num_threads=4,
    loss="bpr",
    learning_rate=0.03,
    no_components=64
)



KHỞI TẠO LIGHTFM RECOMMENDATION SYSTEM
✅ Khởi tạo thành công!

📄 Chuẩn bị dataset cho LightFM...
✅ Dataset chuẩn bị xong!
   - Interactions shape: (1517, 1517)
   - User features shape: (1517, 1575)
   - Item features shape: (1517, 1575)
   - Tổng interactions: 38856

🎓 Training LightFM model...
   - Loss: bpr
   - Epochs: 40
   - Components: 64
   - Learning rate: 0.03
✅ Train xong!


In [133]:
# Build test_interactions từ test_likes (implicit feedback = 1.0)
test_interactions, _ = lightfm_rcm.dataset.build_interactions(
    [(row["liker_id"], row["liked_id"], 1.0) for _, row in test_likes.iterrows()]
)

# Đánh giá
lightfm_metrics = lightfm_rcm.evaluate(test_interactions, k=10)
lightfm_metrics



📊 Đánh giá LightFM model (k=10)...
✅ Kết quả đánh giá:
   - precision@10: 0.0096
   - recall@10: 0.0140
   - auc: 0.7268


{'precision@10': 0.00958721712231636,
 'recall@10': 0.013973060893833199,
 'auc': 0.7267699241638184}

# Lưu lại mô hình

In [134]:
import os
import json
import time
from pathlib import Path

import joblib
import numpy as np

from scipy.sparse import save_npz, load_npz

In [136]:
SAVE_ROOT = "/content/drive/MyDrive/Rcm-sys"

In [137]:
def _now_tag():
    return time.strftime("%Y%m%d-%H%M%S")


def save_recsys_pack(
    save_root: str = SAVE_ROOT,
    rcm_system=None,
    lightfm_rcm=None,
    extra: dict | None = None
):
    """
    Save chuẩn theo kiến trúc notebook của bạn:
    - rcm_system: chứa CF + CBF + Hybrid (methods)
    - lightfm_rcm: LightFM wrapper + model + dataset + features
    - lưu thêm matrices npz (user/item features, train_interactions, weights) để chắc chắn restore được
    """
    save_root = Path(save_root)
    save_root.mkdir(parents=True, exist_ok=True)

    pack_dir = save_root / f"recsys_pack_{_now_tag()}"
    pack_dir.mkdir(parents=True, exist_ok=True)

    meta = {
        "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "artifacts": {},
        "notes": []
    }

    if rcm_system is None:
        # auto pick from globals
        if "rcm_system" in globals():
            rcm_system = globals()["rcm_system"]
        elif "rcm_system" in locals():
            rcm_system = locals()["rcm_system"]

    if rcm_system is None:
        meta["notes"].append("rcm_system is None -> not saved")
    else:
        try:
            joblib.dump(rcm_system, pack_dir / "rcm_system.joblib")
            meta["artifacts"]["rcm_system"] = "rcm_system.joblib"
        except Exception as e:
            meta["notes"].append(f"FAILED saving rcm_system.joblib: {repr(e)}")

    if lightfm_rcm is None:
        if "lightfm_rcm" in globals():
            lightfm_rcm = globals()["lightfm_rcm"]

    if lightfm_rcm is None:
        meta["notes"].append("lightfm_rcm is None -> not saved")
    else:
        # 2a) Try save full wrapper
        try:
            joblib.dump(lightfm_rcm, pack_dir / "lightfm_rcm.joblib")
            meta["artifacts"]["lightfm_rcm"] = "lightfm_rcm.joblib"
        except Exception as e:
            meta["notes"].append(f"FAILED saving lightfm_rcm.joblib: {repr(e)}")
            # fallback: save only model
            try:
                if hasattr(lightfm_rcm, "model"):
                    joblib.dump(lightfm_rcm.model, pack_dir / "lightfm_model_only.joblib")
                    meta["artifacts"]["lightfm_model_only"] = "lightfm_model_only.joblib"
            except Exception as e2:
                meta["notes"].append(f"FAILED fallback saving lightfm_model_only.joblib: {repr(e2)}")

        # 2b) Save matrices (đảm bảo load lại không lệch feature)
        try:
            if hasattr(lightfm_rcm, "user_features"):
                save_npz(pack_dir / "lightfm_user_features.npz", lightfm_rcm.user_features.tocsr())
                meta["artifacts"]["lightfm_user_features"] = "lightfm_user_features.npz"

            if hasattr(lightfm_rcm, "item_features"):
                save_npz(pack_dir / "lightfm_item_features.npz", lightfm_rcm.item_features.tocsr())
                meta["artifacts"]["lightfm_item_features"] = "lightfm_item_features.npz"

            if hasattr(lightfm_rcm, "train_interactions"):
                save_npz(pack_dir / "lightfm_train_interactions.npz", lightfm_rcm.train_interactions.tocsr())
                meta["artifacts"]["lightfm_train_interactions"] = "lightfm_train_interactions.npz"

            if hasattr(lightfm_rcm, "weights") and lightfm_rcm.weights is not None:
                save_npz(pack_dir / "lightfm_weights.npz", lightfm_rcm.weights.tocsr())
                meta["artifacts"]["lightfm_weights"] = "lightfm_weights.npz"
        except Exception as e:
            meta["notes"].append(f"FAILED saving lightfm matrices: {repr(e)}")

        # 2c) Save list user_ids (debug + mapping)
        try:
            if hasattr(lightfm_rcm, "profiles") and lightfm_rcm.profiles is not None:
                user_ids = lightfm_rcm.profiles["id"].astype(str).tolist()
                with open(pack_dir / "lightfm_user_ids.json", "w", encoding="utf-8") as f:
                    json.dump(user_ids, f, ensure_ascii=False)
                meta["artifacts"]["lightfm_user_ids"] = "lightfm_user_ids.json"
        except Exception as e:
            meta["notes"].append(f"FAILED saving lightfm_user_ids.json: {repr(e)}")

    if extra is None:
        extra = {}
        if "lightfm_metrics" in globals():
            extra["lightfm_metrics"] = globals()["lightfm_metrics"]
    if extra:
        try:
            joblib.dump(extra, pack_dir / "extra.joblib")
            meta["artifacts"]["extra"] = "extra.joblib"
        except Exception as e:
            meta["notes"].append(f"FAILED saving extra.joblib: {repr(e)}")

    with open(pack_dir / "meta.json", "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    print("✅ Saved recsys pack to:")
    print(pack_dir)
    print("Files:", os.listdir(pack_dir))

    return str(pack_dir)

In [138]:

pack_dir = save_recsys_pack(
    save_root=SAVE_ROOT,
    rcm_system=rcm_system if "rcm_system" in globals() else None,
    lightfm_rcm=lightfm_rcm if "lightfm_rcm" in globals() else None,
    extra={"k_eval": 10, "lightfm_metrics": lightfm_metrics if "lightfm_metrics" in globals() else None}
)

✅ Saved recsys pack to:
/content/drive/MyDrive/Rcm-sys/recsys_pack_20260126-071707
Files: ['rcm_system.joblib', 'lightfm_rcm.joblib', 'lightfm_user_features.npz', 'lightfm_item_features.npz', 'lightfm_train_interactions.npz', 'lightfm_weights.npz', 'lightfm_user_ids.json', 'extra.joblib', 'meta.json']


# Load model và chạy lại

In [139]:
import joblib
import json
from pathlib import Path
from scipy.sparse import load_npz

def load_rcm_models(pack_dir: str):
    pack_dir = Path(pack_dir)

    # ---- RCM SYSTEM (CONTENT / COLLAB / HYBRID)
    rcm_system = joblib.load(pack_dir / "rcm_system.joblib")

    # ---- LIGHTFM MODEL (MODEL #4)
    lightfm_rcm = None
    lf_path = pack_dir / "lightfm_rcm.joblib"
    if lf_path.exists():
        lightfm_rcm = joblib.load(lf_path)

        # FIX đúng theo freelance.py
        if hasattr(lightfm_rcm, "user_features"):
            lightfm_rcm.user_features_matrix = lightfm_rcm.user_features
        if hasattr(lightfm_rcm, "item_features"):
            lightfm_rcm.item_features_matrix = lightfm_rcm.item_features

        # gắn lại matrix nếu thiếu
        if getattr(lightfm_rcm, "user_features", None) is None:
            lightfm_rcm.user_features = load_npz(pack_dir / "lightfm_user_features.npz")
            lightfm_rcm.user_features_matrix = lightfm_rcm.user_features

        if getattr(lightfm_rcm, "item_features", None) is None:
            lightfm_rcm.item_features = load_npz(pack_dir / "lightfm_item_features.npz")
            lightfm_rcm.item_features_matrix = lightfm_rcm.item_features

        if getattr(lightfm_rcm, "train_interactions", None) is None:
            lightfm_rcm.train_interactions = load_npz(pack_dir / "lightfm_train_interactions.npz")

    return rcm_system, lightfm_rcm


In [140]:
def test_rcm_for_user(
    user_id: str,
    rcm_system,
    lightfm_rcm=None,
    k: int = 10
):
    user_id = str(user_id)

    print("\n" + "="*60)
    print(f"🎯 TEST RECOMMENDATION FOR USER: {user_id}")
    print("="*60)

    # -------- CONTENT
    print("\n🧠 CONTENT-BASED:")
    content_recs = rcm_system.get_content_recommendations(
        user_id,
        top_n=k
    )
    print(content_recs)

    # -------- COLLAB
    print("\n🤝 COLLABORATIVE:")
    collab_recs = rcm_system.get_collaborative_recommendations(
        user_id,
        top_n=k
    )
    print(collab_recs)

    # -------- HYBRID
    print("\n🔀 HYBRID:")
    hybrid_recs = rcm_system.get_hybrid_recommendations(
        user_id,
        top_n=k
    )
    print(hybrid_recs)

    # -------- LIGHTFM
    if lightfm_rcm is not None:
        print("\n🤖 LIGHTFM (MF MODEL):")
        lf_recs = lightfm_rcm.predict_for_user(
            user_id,
            top_n=k
        )
        print(lf_recs)
    else:
        print("\n🤖 LIGHTFM: ❌ not loaded")

    return {
        "content": content_recs,
        "collaborative": collab_recs,
        "hybrid": hybrid_recs,
        "lightfm": lf_recs if lightfm_rcm else None
    }


In [ ]:
pack_dir = "/content/drive/MyDrive/Rcm-sys/recsys_pack_20260127-160021"

rcm_system, lightfm_rcm = load_rcm_models(pack_dir)

# chọn user thật có trong data
USER_ID = "1"

results = test_rcm_for_user(
    user_id=USER_ID,
    rcm_system=rcm_system,
    lightfm_rcm=lightfm_rcm,
    k=10
)



🎯 TEST RECOMMENDATION FOR USER: 1

🧠 CONTENT-BASED:
❌ User 1 không tồn tại!
Empty DataFrame
Columns: []
Index: []

🤝 COLLABORATIVE:
❌ User 1 không có interaction data!
Empty DataFrame
Columns: []
Index: []

🔀 HYBRID:
❌ User 1 không tồn tại!
❌ User 1 không có interaction data!
❌ Không thể tạo recommendations cho user 1
Empty DataFrame
Columns: []
Index: []

🤖 LIGHTFM (MF MODEL):
⚠️ User 1 không có trong dataset
Empty DataFrame
Columns: []
Index: []


In [143]:
rcm_system.profiles["id"].head(20).tolist()

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]